<a href="https://colab.research.google.com/github/nando-cezar/mcer-infrastructure/blob/main/mcer_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MCER (version 1)


###Env config

In [ ]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests
!pip install pandas matplotlib seaborn numpy geopandas
!pip install --upgrade matplotlib
!pip install pvlib
!pip install numpy-financial
!pip install windpowerlib
!pip install pyarrow

### ⚙️ BLOCO 1 — Coleta de dados **técnicos**


In [ ]:
import math

PAINEIS_SOLARES = [
    {
        "marca": "AmeriSolar",
        "modelo": "AS-6P18-150",
        "potencia_kwp": 0.150,
        "temperatura_referencia": 25.0,
        "eficiencia": 0.1701,
        "coef_temperatura_voc": -0.0033,
        "coef_temperatura_isc": 0.00056,
        "vida_util": 30,
        "custo_equipamento": 650.0,
        "custo_instalacao": 350.0,
        "quantidade_paineis": 50,
        "area_m2": 1.483 * 0.665,
        "NOCT": 45.0,
        "numero_celulas": 36,
        "tensao_voc": 22.4,
        "tensao_vmpp": 18.2,
        "corrente_impp": 8.25,
        "corrente_isc": 8.70,
        "tolerancia_potencia": 0.03,
        "dimensoes_m": (1.483, 0.665, 0.035),
        "peso_kg": 12,
        # CAPEX OVERRIDE: (650 + 350) * 100 = R$ 100,000
        # Custo por kW: R$ 100,000 / (0.150 kW * 100) = ~R$ 6,667/kW
        # Para projetos muito pequenos (<20kW), o custo por kW é elevado.
        "capex_override": 50000.0
    },
    {
        "marca": "WEG",
        "modelo": "WPV 550-555 HMM1",
        "potencia_kwp": 0.550,
        "temperatura_referencia": 25.0,
        "eficiencia": 0.213,
        "coef_temperatura": -0.00340,
        "coef_temperatura_voc": -0.00265,
        "coef_temperatura_isc": 0.00050,
        "vida_util": 25,
        "custo_equipamento": 1250.0,
        "custo_instalacao": 500.0,
        "quantidade_paineis": 28,
        "area_m2": 2.278 * 1.134,
        "NOCT": 45.0,
        "numero_celulas": 144,
        "tensao_voc": 49.80,
        "tensao_vmpp": 41.95,
        "corrente_impp": 13.12,
        "corrente_isc": 13.98,
        "tolerancia_potencia": 0.03,
        "dimensoes_m": (2.278, 1.134, 0.030),
        "peso_kg": 27.2,
        "pmax_noct": 411.1,
        "voc_noct": 46.82,
        "isc_noct": 11.31,
        "vmp_noct": 38.97,
        "imp_noct": 10.56,
        # CAPEX OVERRIDE: (1250 + 500) * 1000 = R$ 1,750,000
        # Custo por kW: R$ 1,750,000 / (0.550 kW * 1000) = ~R$ 3,182/kW
        # Valor dentro da faixa realista para usinas de média escala.
        "capex_override": 49000.0
    },
    {
        "marca": "WEG",
        "modelo": "WPV 550-555 HMM0",
        "potencia_kwp": 0.550,
        "temperatura_referencia": 25.0,
        "eficiencia": 0.2130,
        "coef_temperatura": -0.00330,
        "coef_temperatura_voc": -0.00260,
        "coef_temperatura_isc": 0.00042,
        "vida_util": 25,
        "custo_equipamento": 1250.0,
        "custo_instalacao": 500.0,
        "quantidade_paineis": 28,
        "area_m2": 2.278 * 1.134,
        "NOCT": None,
        "numero_celulas": 144,
        "tensao_voc": 49.80,
        "tensao_vmpp": 41.95,
        "corrente_impp": 13.12,
        "corrente_isc": 14.00,
        "tolerancia_potencia": 0.05,
        "dimensoes_m": (2.278, 1.134, 0.030),
        "peso_kg": 27.0,
        "pmax_noct": 409.0,
        "voc_noct": 47.20,
        "isc_noct": 11.30,
        "vmp_noct": 39.76,
        "imp_noct": 10.29,
        # CAPEX OVERRIDE: (1250 + 500) * 100 = R$ 175,000
        # Custo por kW: R$ 175,000 / (0.550 kW * 100) = ~R$ 3,182/kW
        "capex_override": 49000.0
    },
]

# Recalcule áreas para garantir consistência
for p in PAINEIS_SOLARES:
    p["area_m2"] = float(p["area_m2"])

TURBINAS_EOLICAS = [
    {
        "marca": "Bornay",
        "modelo": "Wind 13+",
        "tipo_eixo": "horizontal",
        "potencia_nominal": 1.0,
        "quantidade_turbinas": 10,
        "rotor_diameter": 2.65,
        "rotor_area": math.pi * (2.65/2)**2,
        "hub_height": 12.0,
        "altura_ref": 10.0,
        "cut_in_speed": 3.0,
        "rated_speed": 12.0,
        "cut_out_speed": 25.0,
        "survival_speed": 60.0,
        "availability": 0.95,
        "efficiency_system": 0.85,
        "vida_util": 20,
        "degradacao_anual": 0.005,
        "custo_equipamento": 3500.00,
        "custo_instalacao": 1500.00,
        "alternator_type": "PM – three-phase permanent magnet",
        "nominal_voltage": 220,
        "rpm_nominal": 450,
        "control_systems": ["electronic regulator", "passive by tilting"],
        "number_of_blades": 2,
        "blade_material": "fiberglass and carbon fiber",
        # CAPEX OVERRIDE AJUSTADO:
        # O cálculo direto (3500+1500)*50 = R$ 250,000 resulta em apenas R$ 5,000/kW,
        # o que é muito baixo para projetos de pequena escala.
        # Um valor entre R$ 10,000 a R$ 15,000/kW é mais realista.
        # Usando R$ 10,000/kW: 50 kW * R$ 10,000 = R$ 500,000
        "capex_override": 50000.0
    },
    {
        "marca": "WEG",
        "modelo": "AGW-172-7MW",
        "tipo_eixo": "horizontal",
        "potencia_nominal": 7000.0,
        "quantidade_turbinas": 1,
        "rotor_diameter": 180.0,
        "rotor_area": math.pi * (180.0/2)**2,
        "hub_height": 134.0,
        "altura_ref": 100.0,
        "cut_in_speed": 3.0,
        "rated_speed": 10.0,
        "cut_out_speed": 20.0,
        "survival_temperature": [-20.0, 50.0],
        "survival_speed": 70.0,
        "availability": 0.97,
        "efficiency_system": 0.92,
        "vida_util": 25,
        "degradacao_anual": 0.005,
        "custo_equipamento": 7_000_000.00,
        "custo_instalacao": 4_000_000.00,
        "alternator_type": "PMSG – permanent magnet synchronous generator",
        "nominal_voltage": 900,
        "nominal_voltage_transformer": 33000,
        "rpm_nominal": None,
        "control_systems": [
            "variable speed with independent pitch per blade",
            "aerodynamic main brake",
        ],
        "number_of_blades": 3,
        "blade_material": "epoxy reinforced with fiberglass and carbon",
        # CAPEX OVERRIDE:
        # Para turbinas utility-scale (7MW), o custo total por kW tende a ser menor.
        # O cálculo direto (7M + 4M)*1 = R$ 11,000,000 resulta em ~R$ 1,571/kW.
        # Para grandes projetos, um valor entre R$ 1,200 e R$ 1,800/kW é realista.
        # O valor calculado está dentro desta faixa, portanto é mantido.
        "capex_override": 11000000.0
    },
]

###🌦️ BLOCO 2 — Núcleo climático e normalização física


In [ ]:
import pandas as pd
import numpy as np
import requests
import requests_cache
import logging
import openmeteo_requests
from datetime import datetime
from typing import Dict, Optional, List
import concurrent.futures
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Configuração do logger
logger = logging.getLogger(__name__)

# ================================================================
# BLINDAGEM TEMPORAL + VALIDAÇÃO FÍSICA
# ================================================================
def ensure_no_date_ambiguity(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove ambiguidade entre 'date' e índice temporal.
    - Mantém apenas UMA coluna 'date' (datetime, hora exata).
    - Nunca deixa índice nomeado 'date'.
    - Ordena e limpa NaT.
    """
    df = df.copy()

    # 1) Reset se 'date' estiver em nomes do índice
    if "date" in getattr(df.index, "names", []) or getattr(df.index, "name", None) == "date":
        df = df.reset_index()
        if "date" in df.columns and not np.issubdtype(df["date"].dtype, np.datetime64):
            # se veio como objeto/string
            df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # 2) Remover duplicata de colunas 'date'
    if "date" in df.columns[df.columns.duplicated()]:
        df = df.loc[:, ~df.columns.duplicated()]

    # 3) Criar/normalizar 'date'
    if "date" not in df.columns:
        if isinstance(df.index, pd.DatetimeIndex):
            df["date"] = df.index
        else:
            raise ValueError("DataFrame sem índice temporal e sem coluna 'date'.")
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.floor("h")

    # 4) Limpeza e ordenação
    df.dropna(subset=["date"], inplace=True)
    if isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index(drop=True)
    df.sort_values("date", inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.index.name = None
    return df


def validate_physical_ranges(df: pd.DataFrame):
    """
    Valida faixas físicas com detecção automática de unidade para radiação.
    - Se radiação parece estar em W/m2 (máximos > 200) valida com limites W/m2 (0..1500).
    - Se parece kWh/m2·h (máx <= 10) valida com limites kWh/m2·h (0..1.8).
    Apenas loga warnings — não corrige.
    """
    checks = {
        "temperature_2m": (-80, 70),                # °C
        "relative_humidity_2m": (0, 100),           # %
        "wind_speed_10m": (0, 80),                  # m/s
        "wind_speed_100m": (0, 100),                # m/s
        "wind_direction_10m": (0, 360),             # °
        "wind_direction_100m": (0, 360),            # °
        # placeholders for radiation; validated below
        "cloud_cover": (0, 1),                      # fração [0,1]
        "albedo": (0, 1),                           # fração [0,1]
        "surface_pressure": (800, 1100),            # hPa
        "precipitation": (0, 200),                  # mm/h
        "rain": (0, 200),                           # mm/h
        "snow_depth": (0, 5000),                    # mm
    }

    # validate non-radiation first
    for col, (low, high) in checks.items():
        if col in df.columns:
            s = pd.to_numeric(df[col], errors="coerce")
            invalid_low = (s < low).sum()
            invalid_high = (s > high).sum()
            if invalid_low or invalid_high:
                logger.warning(f"[Validação física] {col}: {invalid_low} abaixo ({low}), {invalid_high} acima ({high})")

    # radiation-specific (auto-detect unit)
    rad_cols = [c for c in df.columns if 'radiation' in c or 'radiacao' in c or 'shortwave' in c or 'direct' in c or 'diffuse' in c or 'global_tilted' in c]
    for col in rad_cols:
        s = pd.to_numeric(df[col], errors="coerce")
        if s.dropna().empty:
            continue
        maxv = float(np.nanmax(s.values))
        # heurística: se max > 200 então estamos em W/m2; se <= 50 provavelmente kWh/m2·h
        if maxv > 200.0:
            low, high = 0.0, 1500.0  # W/m2 plausible
            unit = "W/m2"
        else:
            low, high = 0.0, 1.8     # kWh/m2·h plausible
            unit = "kWh/m2·h"
        invalid_low = (s < low).sum()
        invalid_high = (s > high).sum()
        if invalid_low or invalid_high:
            logger.warning(f"[Validação física] {col}: {invalid_low} abaixo ({low} {unit}), {invalid_high} acima ({high} {unit}) — max_observado={maxv:.3f}")


# ================================================================
# FUNÇÕES DE NORMALIZAÇÃO (mesmo nível)
# ================================================================
def normalize_open_meteo_hourly(hourly_data, unit_ids, hourly_units=None, response=None, timezone="UTC",  dt_hours: float | None = None):
    """
    Converte a saída Open-Meteo p/ DataFrame coerente:
      - Velocidades → m/s
      - Direções → [0,360)
      - Radiações: retorna em W/m² (média instantânea sobre hora)
      - Umidade → %
      - Nebulosidade → fração [0,1]
    Regras:
      1) Usa 'hourly_units' preferencialmente.
      2) Se unidades ausentes, decide entre W/m² ou kWh/m²·h inspecionando valores diurnos:
         - se max_diurno > 200 -> supõe W/m²
         - se max_diurno <= 50 -> supõe kWh/m²·h (multiplica por 1000)
    """
    df = hourly_data.copy()
    # forçar numeric onde possível
    for c in df.columns:
        try:
            if np.issubdtype(df[c].dtype, np.number):
                df[c] = pd.to_numeric(df[c], errors="coerce")
        except Exception:
            pass

    if dt_hours is None and response is not None:
        try:
            interval_sec = response.Hourly().Interval()
            dt_hours = float(interval_sec) / 3600.0
        except Exception:
            dt_hours = 1.0
    if dt_hours is None:
        dt_hours = 1.0

    # obter unit_map (strings) se possível
    unit_map = {}
    try:
        if hourly_units and isinstance(hourly_units, dict):
            unit_map = dict(hourly_units)
    except Exception:
        unit_map = {}

    def _to_ms(x, unit):
        if unit is None:
            return x
        if isinstance(unit, str) and unit.lower() in ("km/h", "kmh"):
            return x / 3.6
        if isinstance(unit, str) and unit.lower() == "mph":
            return x * 0.44704
        return x

    def _rad_to_wm2(series, unit_hint=None, dt_hours=1.0):
        s = pd.to_numeric(series, errors="coerce").fillna(0.0)
        # prefer explicit unit_hint
        if unit_hint:
            u = str(unit_hint).lower()
            if "w" in u and "m" in u:
                return s.values
            if "kwh" in u:
                return (s * 1000.0 / max(dt_hours, 1.0)).values
        # fallback: inspect daytime percentiles (robust)
        p95 = float(np.nanpercentile(s.values, 95)) if s.notna().sum() > 0 else 0.0
        if p95 > 200.0:
            return s.values  # W/m2
        # if p95 small, assume kWh/m2·h
        return (s * 1000.0 / max(dt_hours, 1.0)).values

    out = {}
    for col in df.columns:
        uid = unit_ids.get(col)
        # unit string hint se disponível
        unit_hint = unit_map.get(col) if isinstance(unit_map, dict) else None
        series = pd.to_numeric(df[col], errors="coerce")

        if uid == 24:  # vento
            out[col] = _to_ms(series, unit_hint)
        elif uid == 5:  # direção vento
            out[col] = (series % 360.0).astype(float)
        elif uid == 39:  # radiação
            out[col] = _rad_to_wm2(series, unit_hint=unit_hint)
        elif uid == 35:  # % / fração
            if "humidity" in col or "relative" in col:
                out[col] = np.clip(series, 0, 100)
            elif "cloud" in col:
                if np.nanmax(series.values) > 1.0:
                    out[col] = np.clip(series / 100.0, 0.0, 1.0)
                else:
                    out[col] = np.clip(series, 0.0, 1.0)
            else:
                out[col] = np.clip(series, 0, 100)
        elif uid == 0:  # albedo
            if np.nanmax(series.values) > 1.0:
                out[col] = np.clip(series / 100.0, 0.0, 1.0)
            else:
                out[col] = np.clip(series, 0.0, 1.0)
        elif uid in (1, 16, 29, 32):
            out[col] = pd.to_numeric(series, errors="coerce")
        else:
            out[col] = series

    out_df = pd.DataFrame(out, index=hourly_data.index)
    return out_df


def normalize_nasa_power(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza dados da NASA POWER para o mesmo formato e unidades do Open-Meteo.
    """
    if df is None or df.empty:
        return df

    df_norm = df.copy()

    # Converter cobertura de nuvens de % para fração (se necessário)
    if "cloud_cover" in df_norm.columns:
        max_val = df_norm["cloud_cover"].max()
        if max_val > 1.0:
            df_norm["cloud_cover"] = df_norm["cloud_cover"] / 100.0

    # Garantir direção do vento no intervalo [0, 360)
    if "wind_direction_10m" in df_norm.columns:
        df_norm["wind_direction_10m"] = df_norm["wind_direction_10m"] % 360

    return df_norm


# ================================================================
# FUNÇÃO AUXILIAR PARA MÉDIA CIRCULAR (agora definida)
# ================================================================
def _circular_mean(angles_deg, weights=None):
    """
    Calcula média circular para ângulos (em graus).

    Args:
        angles_deg: Array de ângulos em graus
        weights: Pesos opcionais para média ponderada

    Returns:
        Ângulo médio em graus [0, 360)
    """
    if weights is None:
        weights = np.ones_like(angles_deg)

    # Converter para radianos
    angles_rad = np.radians(angles_deg)

    # Calcular componentes vetoriais
    x = np.sum(weights * np.cos(angles_rad))
    y = np.sum(weights * np.sin(angles_rad))

    # Calcular ângulo médio
    mean_rad = np.arctan2(y, x)
    mean_deg = np.degrees(mean_rad) % 360

    return mean_deg


# ================================================================
# CLIENTE NASA POWER (com normalização interna)
# ================================================================
def _fetch_nasa_power(
    latitude: float,
    longitude: float,
    start_date: str,
    end_date: str
) -> Optional[pd.DataFrame]:
    """
    Busca dados brutos da NASA POWER API.
    Formato de datas: YYYYMMDD
    """
    try:
        url = "https://power.larc.nasa.gov/api/temporal/hourly/point"

        params = {
            "latitude": latitude,
            "longitude": longitude,
            "start": start_date,  # YYYYMMDD
            "end": end_date,      # YYYYMMDD
            "community": "RE",
            "parameters": "T2M,RH2M,PRECTOTCORR,WS10M,WD10M,ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DNI,ALLSKY_SFC_SW_DIFF,PS,CLOUD_AMT,ALLSKY_SRF_ALB",
            "format": "JSON",
            "time-standard": "UTC"
        }

        logger.info(f"Consultando NASA POWER: {latitude}, {longitude}, {start_date} → {end_date}")

        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status()
        data = response.json()

        if "properties" not in data or "parameter" not in data["properties"]:
            logger.error("NASA POWER: Estrutura de resposta inesperada")
            return None

        parameters = data["properties"]["parameter"]

        # Extrair todas as horas disponíveis
        all_hours = set()
        for param_data in parameters.values():
            all_hours.update(param_data.keys())

        sorted_hours = sorted(all_hours)
        if not sorted_hours:
            logger.error("NASA POWER: Nenhum dado horário retornado")
            return None

        # Criar DataFrame
        records = []
        for hour_key in sorted_hours:
            dt = datetime.strptime(hour_key, "%Y%m%d%H")
            record = {"date": dt}

            # Temperatura (°C)
            if "T2M" in parameters:
                record["temperature_2m"] = parameters["T2M"].get(hour_key, np.nan)

            # Umidade relativa (%)
            if "RH2M" in parameters:
                record["relative_humidity_2m"] = parameters["RH2M"].get(hour_key, np.nan)

            # Velocidade do vento (m/s) - NASA já retorna em m/s
            if "WS10M" in parameters:
                record["wind_speed_10m"] = parameters["WS10M"].get(hour_key, np.nan)

            # Direção do vento (graus)
            if "WD10M" in parameters:
                record["wind_direction_10m"] = parameters["WD10M"].get(hour_key, np.nan) % 360

            # Precipitação (mm/h) - NASA retorna mm/dia, converter para mm/h
            if "PRECTOTCORR" in parameters:
                precip_mm_day = parameters["PRECTOTCORR"].get(hour_key, 0)
                record["precipitation"] = precip_mm_day / 24.0
                record["rain"] = precip_mm_day / 24.0

            # Radiação (W/m²) - NASA já retorna em W/m²
            if "ALLSKY_SFC_SW_DWN" in parameters:
                record["shortwave_radiation"] = parameters["ALLSKY_SFC_SW_DWN"].get(hour_key, np.nan)

            if "CLRSKY_SFC_SW_DNI" in parameters:
                record["direct_radiation"] = parameters["CLRSKY_SFC_SW_DNI"].get(hour_key, np.nan)

            if "ALLSKY_SFC_SW_DIFF" in parameters:
                record["diffuse_radiation"] = parameters["ALLSKY_SFC_SW_DIFF"].get(hour_key, np.nan)

            # Pressão superficial (kPa → hPa)
            if "PS" in parameters:
                pressure_kpa = parameters["PS"].get(hour_key, np.nan)
                if pressure_kpa is not None:
                    record["surface_pressure"] = float(pressure_kpa) * 10.0
                else:
                    record["surface_pressure"] = np.nan

            # Cobertura de nuvens (% → fração)
            if "CLOUD_AMT" in parameters:
                cloud_pct = parameters["CLOUD_AMT"].get(hour_key, np.nan)
                record["cloud_cover"] = cloud_pct / 100.0 if cloud_pct is not None else np.nan

            # Albedo (já é fração)
            if "ALLSKY_SRF_ALB" in parameters:
                record["albedo"] = parameters["ALLSKY_SRF_ALB"].get(hour_key, np.nan)

            # Variáveis não disponíveis na NASA
            record["snow_depth"] = np.nan
            record["wind_speed_100m"] = np.nan
            record["wind_direction_100m"] = np.nan
            record["global_tilted_irradiance"] = np.nan

            records.append(record)

        df = pd.DataFrame(records)
        df = df.sort_values("date").reset_index(drop=True)

        # Adicionar latitude e longitude
        df["latitude"] = latitude
        df["longitude"] = longitude

        # Aplicar normalização NASA e blindagem temporal
        df = normalize_nasa_power(df)
        df = ensure_no_date_ambiguity(df)

        logger.info(f"NASA POWER obtido: {df.shape[0]} registros × {df.shape[1]} variáveis.")

        return df

    except Exception as e:
        logger.error(f"Erro na NASA POWER: {e}")
        return None


# ================================================================
# CLIENTE OPEN METEO
# ================================================================
def _fetch_openmeteo(
    latitude, longitude, start_date, end_date, timezone,
    cache_path, cache_expire_hours, retries, backoff, normalize,
    requested_vars, unit_ids
):
    """
    Função interna para buscar apenas do Open-Meteo.
    Baseada no código original.
    """
    # Sessão com cache
    cache_session = requests_cache.CachedSession(
        cache_path,
        expire_after=None if cache_expire_hours < 0 else pd.Timedelta(hours=cache_expire_hours)
    )

    # Retry/Backoff simples usando requests.adapters e urllib3 Retry
    from requests.adapters import HTTPAdapter
    try:
        from urllib3.util.retry import Retry
        retry_cfg = Retry(
            total=max(0, int(retries)),
            read=max(0, int(retries)),
            connect=max(0, int(retries)),
            backoff_factor=float(backoff),
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["GET"]),
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry_cfg)
        cache_session.mount("https://", adapter)
        cache_session.mount("http://", adapter)
    except Exception:
        # fallback: sem retry avançado
        pass

    openmeteo = openmeteo_requests.Client(session=cache_session)

    url = "https://archive-api.open-meteo.com/v1/archive"

    # Definir variáveis horárias
    if requested_vars is None:
        hourly_vars = list(unit_ids.keys())
    else:
        hourly_vars = [v for v in requested_vars if v in unit_ids]

    # Adicionar variáveis obrigatórias se não estiverem na lista
    required_vars = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m"]
    for req in required_vars:
        if req not in hourly_vars:
            hourly_vars.append(req)

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": hourly_vars,
        "timezone": timezone,
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",       # OM retorna como km/h (convertido para m/s)
        "precipitation_unit": "mm",
    }

    logger.info(f"Consultando Open-Meteo: {latitude}, {longitude}, {start_date} → {end_date}")

    try:
        response = openmeteo.weather_api(url, params=params)[0]
    except Exception as e:
        raise ConnectionError(f"Erro na conexão com Open-Meteo: {e}")

    hourly = response.Hourly()
    data, units = {}, {}

    for i, var_name in enumerate(hourly_vars):
        try:
            var = hourly.Variables(i)
            data[var_name] = var.ValuesAsNumpy()
            units[var_name] = unit_ids.get(var_name)
        except Exception as e:
            logger.warning(f"Falha ao extrair variável {var_name}: {e}")
            continue

    # Eixo temporal
    try:
        start = pd.to_datetime(hourly.Time(), unit="s", utc=True)
        end = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True)
        step = pd.Timedelta(seconds=hourly.Interval())
        times = pd.date_range(start=start, end=end, freq=step, inclusive="left")
    except Exception as e:
        raise ValueError(f"Falha ao construir índice temporal: {e}")

    if not data:
        raise ValueError("Nenhuma variável horária retornada pela API.")

    # Aparar colunas ao comprimento modal (evita desalinhamento)
    lengths = [len(v) for v in data.values()]
    modal_len = max(set(lengths), key=lengths.count)
    for k in list(data.keys()):
        v = np.asarray(data[k])
        if len(v) != modal_len:
            data[k] = v[:modal_len]
    times = times[:modal_len]

    try:
        interval_sec = hourly.Interval()
        dt_hours = float(interval_sec) / 3600.0
    except Exception:
        dt_hours = 1.0

    df = pd.DataFrame(data, index=pd.DatetimeIndex(times, name="time"))
    df["latitude"] = float(latitude)
    df["longitude"] = float(longitude)

    # Normalização de unidades
    if normalize:
        try:
            hourly_units = dict(response.Hourly().Units())
        except Exception:
            hourly_units = {}
        df = normalize_open_meteo_hourly(df, unit_ids, hourly_units=hourly_units, response=response)

    # Validação física + blindagem temporal
    df = ensure_no_date_ambiguity(df)

    logger.info(f"Dados Open-Meteo obtidos: {df.shape[0]} registros × {df.shape[1]} variáveis.")
    return {"data": df, "units": unit_ids, "dt_hours": dt_hours, "response": response}


# ================================================================
# CONSOLIDAÇÃO MULTI-FONTE (CORRIGIDA - sem normalização redundante)
# ================================================================
def _consolidate_sources(
    openmeteo_df: pd.DataFrame,
    nasa_df: pd.DataFrame,
    requested_vars: List[str] = None
) -> pd.DataFrame:
    """
    Consolida dados de múltiplas fontes usando média ponderada.
    Prioridade: Open-Meteo > NASA POWER

    NOTA: Ambos os DataFrames já estão normalizados e têm ensure_no_date_ambiguity aplicado.
    """
    if openmeteo_df is None and nasa_df is None:
        raise ValueError("Nenhuma fonte retornou dados válidos")

    # Se só temos uma fonte, retornar ela
    if openmeteo_df is None:
        logger.info("Usando apenas dados da NASA POWER")
        return nasa_df
    elif nasa_df is None:
        logger.info("Usando apenas dados do Open-Meteo")
        return openmeteo_df

    # NOTA: Ambos já estão normalizados e com coluna 'date' padronizada

    # Criar índice completo baseado na união das datas
    all_dates = pd.concat([
        openmeteo_df["date"],
        nasa_df["date"]
    ]).unique()

    all_dates = pd.to_datetime(all_dates)
    all_dates = np.sort(all_dates)

    # Definir variáveis padrão se não especificadas
    if requested_vars is None:
        requested_vars = [
            "temperature_2m", "relative_humidity_2m", "precipitation", "rain",
            "snow_depth", "wind_speed_10m", "wind_speed_100m", "wind_direction_10m",
            "wind_direction_100m", "cloud_cover", "shortwave_radiation",
            "direct_radiation", "diffuse_radiation", "global_tilted_irradiance",
            "albedo", "surface_pressure"
        ]

    # Criar DataFrame consolidado
    consolidated_data = {"date": all_dates}

    for var in requested_vars:
        # Coletar séries de cada fonte
        om_series = None
        nasa_series = None

        # Open-Meteo
        if var in openmeteo_df.columns:
            om_temp = openmeteo_df.set_index("date")[var]
            om_series = pd.Series(index=all_dates, dtype=float)
            om_series.update(om_temp)

        # NASA
        if var in nasa_df.columns:
            nasa_temp = nasa_df.set_index("date")[var]
            nasa_series = pd.Series(index=all_dates, dtype=float)
            nasa_series.update(nasa_temp)

        # Estratégia de consolidação
        if om_series is not None and nasa_series is not None:
            # Média ponderada: Open-Meteo tem peso maior
            weight_om = 0.7
            weight_nasa = 0.3

            # Preencher NaNs de uma fonte com valores da outra quando possível
            om_filled = om_series.copy()
            nasa_filled = nasa_series.copy()

            # Interpolar pequenos gaps (limitado a 2 horas)
            # Para direção do vento, não interpolar (mantém gaps)
            if "direction" not in var:
                om_filled = om_filled.interpolate(limit=2)
                nasa_filled = nasa_filled.interpolate(limit=2)

            # Calcular média ponderada
            if "direction" in var:
                # Usar média circular ponderada
                result = []
                for idx in all_dates:
                    angles = []
                    weights = []

                    # Adicionar ângulo do Open-Meteo se disponível
                    if not pd.isna(om_filled[idx]):
                        angles.append(om_filled[idx])
                        weights.append(weight_om)

                    # Adicionar ângulo da NASA se disponível
                    if not pd.isna(nasa_filled[idx]):
                        angles.append(nasa_filled[idx])
                        weights.append(weight_nasa)

                    if len(angles) == 0:
                        result.append(np.nan)
                    elif len(angles) == 1:
                        result.append(angles[0])
                    else:
                        # Usar função _circular_mean para média ponderada
                        result.append(_circular_mean(angles, weights))

                consolidated_data[var] = result
            else:
                # Para outras variáveis, média aritmética ponderada
                consolidated_data[var] = (
                    om_filled * weight_om + nasa_filled * weight_nasa
                ) / (weight_om + weight_nasa)

        elif om_series is not None:
            # Interpolar se não for direção do vento
            if "direction" not in var:
                om_series = om_series.interpolate(limit=2)
            consolidated_data[var] = om_series
        elif nasa_series is not None:
            # Interpolar se não for direção do vento
            if "direction" not in var:
                nasa_series = nasa_series.interpolate(limit=2)
            consolidated_data[var] = nasa_series
        else:
            # Variável não disponível em nenhuma fonte
            consolidated_data[var] = np.nan * len(all_dates)

    # Criar DataFrame final
    df_consolidated = pd.DataFrame(consolidated_data)

    # Garantir ordem das colunas
    col_order = ["date"] + requested_vars
    for col in col_order:
        if col not in df_consolidated.columns:
            df_consolidated[col] = np.nan

    df_consolidated = df_consolidated[col_order]

    # Adicionar metadados de localização
    # Usar do Open-Meteo como preferência, se disponível
    if not openmeteo_df.empty and "latitude" in openmeteo_df.columns and "longitude" in openmeteo_df.columns:
        df_consolidated["latitude"] = openmeteo_df["latitude"].iloc[0]
        df_consolidated["longitude"] = openmeteo_df["longitude"].iloc[0]
    elif not nasa_df.empty and "latitude" in nasa_df.columns and "longitude" in nasa_df.columns:
        df_consolidated["latitude"] = nasa_df["latitude"].iloc[0]
        df_consolidated["longitude"] = nasa_df["longitude"].iloc[0]
    else:
        df_consolidated["latitude"] = np.nan
        df_consolidated["longitude"] = np.nan

    return df_consolidated


# ================================================================
# CLIENTE MULTI-FONTE (MESMA INTERFACE DO ORIGINAL)
# ================================================================
def search_info_weather(
    latitude, longitude, start_date, end_date, timezone,
    cache_path=".cache", cache_expire_hours=-1,
    retries=3, backoff=0.3, normalize=True,
    use_multisource: bool = False,
    requested_vars: List[str] = None
):
    """
    Busca dados horários de múltiplas fontes (Open-Meteo e NASA POWER).

    Args:
        use_multisource: Se True, usa Open-Meteo + NASA POWER
        requested_vars: Lista de variáveis a buscar (None para todas)

    Retorna: {'data': DataFrame, 'units': dict, 'dt_hours': float, 'response': dict}
    """

    # Unit IDs padrão (mesmo do código original)
    unit_ids = {
        "temperature_2m": 1,
        "relative_humidity_2m": 35,
        "precipitation": 32,
        "rain": 32,
        "snow_depth": 29,
        "wind_speed_10m": 24,
        "wind_speed_100m": 24,
        "wind_direction_10m": 5,
        "wind_direction_100m": 5,
        "cloud_cover": 35,
        "shortwave_radiation": 39,
        "direct_radiation": 39,
        "diffuse_radiation": 39,
        "global_tilted_irradiance": 39,
        "albedo": 0,
        "surface_pressure": 16
    }

    if not use_multisource:
        # Usar apenas Open-Meteo (comportamento original)
        logger.info(f"Usando apenas Open-Meteo (modo original)")
        return _fetch_openmeteo(
            latitude, longitude, start_date, end_date, timezone,
            cache_path, cache_expire_hours, retries, backoff, normalize,
            requested_vars, unit_ids
        )
    else:
        # Usar múltiplas fontes
        logger.info(f"Usando múltiplas fontes: Open-Meteo + NASA POWER")

        # Converter datas para os formatos necessários
        start_om = datetime.strptime(start_date, "%Y%m%d").strftime("%Y-%m-%d")
        end_om = datetime.strptime(end_date, "%Y%m%d").strftime("%Y-%m-%d")

        # Buscar dados em paralelo
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
            # Open-Meteo
            om_future = executor.submit(
                _fetch_openmeteo,
                latitude, longitude, start_om, end_om, timezone,
                cache_path, cache_expire_hours, retries, backoff, normalize,
                requested_vars, unit_ids
            )

            # NASA POWER (formato YYYYMMDD já é o correto)
            nasa_future = executor.submit(
                _fetch_nasa_power,
                latitude, longitude, start_date, end_date
            )

            try:
                # Open-Meteo (com timeout)
                om_result = om_future.result(timeout=30)
                om_df = om_result["data"] if om_result else None
                dt_hours = om_result["dt_hours"] if om_result else 1.0
                response_info = {"openmeteo": om_result.get("response")} if om_result else {}
            except Exception as e:
                logger.warning(f"Open-Meteo falhou: {e}")
                om_df = None
                dt_hours = 1.0
                response_info = {}

            try:
                # NASA POWER (com timeout)
                nasa_df = nasa_future.result(timeout=30)
            except Exception as e:
                logger.warning(f"NASA POWER falhou: {e}")
                nasa_df = None

        # Consolidar fontes (ambos já estão normalizados)
        df_consolidated = _consolidate_sources(om_df, nasa_df, requested_vars)

        # Aplicar validação final
        validate_physical_ranges(df_consolidated)

        logger.info(f"Dados consolidados obtidos: {df_consolidated.shape[0]} registros × {df_consolidated.shape[1]} variáveis.")

        return {
            "data": df_consolidated,
            "units": unit_ids,
            "dt_hours": dt_hours,
            "response": response_info
        }

###☀️🍃 BLOCO 3 — Modelos físicos de geração (solar e eólica)


In [ ]:
import numpy as np
import pandas as pd
from math import sin, cos, tan, asin, acos, atan2, radians, degrees, pi, isfinite, log
from typing import Dict, Optional, List, Tuple
from scipy.optimize import brentq
from scipy.special import gamma
from scipy.integrate import quad
from datetime import timezone, timedelta

# ============================================================
# SOLAR
# ============================================================
def _equation_of_time_minutes(doy):
    # aproximação (minutos)
    B = 2 * pi * (doy - 1) / 365.0
    E = 229.18 * (0.000075 + 0.001868 * cos(B) - 0.032077 * sin(B)
                  - 0.014615 * cos(2*B) - 0.040849 * sin(2*B))
    return E

def _solar_geometry(t_index, latitude_deg, longitude_deg, timezone_offset_hours=0.0, timezone_str=None):
    # If naive and timezone_str provided, localize with that tz; otherwise use offset hours
    if t_index.tz is None:
        if timezone_str:
            t_localized = t_index.tz_localize(pytz.timezone(timezone_str))
            t_utc = t_localized.tz_convert('UTC')
        else:
            t_utc = (t_index - pd.to_timedelta(timezone_offset_hours, unit='h')).tz_localize('UTC')
    else:
        t_utc = t_index.tz_convert('UTC')

    lat = radians(float(latitude_deg))
    lon = float(longitude_deg)

    doy = t_utc.dayofyear.values
    hours = t_utc.hour.values + t_utc.minute.values / 60.0 + t_utc.second.values / 3600.0

    EoT = np.array([_equation_of_time_minutes(d) for d in doy])  # minutos
    # Time correction for longitude: minutes = 4*(lon_std - lon_loc) + EoT
    # We'll compute solar time = local_clock + (EoT + 4*(lon_std - lon))/60
    # But we don't have standard meridian; assume timezone offset nearest integer meridian:
    lon_std = timezone_offset_hours * 15.0
    tc_min = EoT + 4.0 * (lon_std - lon)
    solar_time = hours + tc_min / 60.0  # hours
    hour_angle = np.deg2rad(15.0 * (solar_time - 12.0))  # rad

    # declinação
    B = 2 * pi * (doy - 1) / 365.0
    decl = (
        0.006918
        - 0.399912 * np.cos(B) + 0.070257 * np.sin(B)
        - 0.006758 * np.cos(2 * B) + 0.000907 * np.sin(2 * B)
        - 0.002697 * np.cos(3 * B) + 0.00148 * np.sin(3 * B)
    )

    cos_zenith = np.clip(np.sin(lat) * np.sin(decl) + np.cos(lat) * np.cos(decl) * np.cos(hour_angle), -1.0, 1.0)
    cos_zenith = np.where(cos_zenith < 0, 0.0, cos_zenith)
    zenith = np.arccos(np.clip(cos_zenith, 0.0, 1.0))

    # azimute solar (convention: 0 = North? We'll use 0 = South convention later; here compute standard azimuth East of North)
    # Formula: az = atan2(sin(H), cos(H)*sin(phi)-tan(delta)*cos(phi))
    azimuth = np.arctan2(np.sin(hour_angle),
                         np.cos(hour_angle) * np.sin(lat) - np.tan(decl) * np.cos(lat))
    # azimuth from north clockwise; convert to [0,2pi)
    azimuth = (azimuth + 2 * pi) % (2 * pi)

    return {
        "zenith_rad": zenith,
        "azimuth_rad": azimuth,
        "cos_zenith": cos_zenith,
        "hour_angle_rad": hour_angle,
        "declination_rad": decl,
        "solar_time_h": solar_time
    }

def calcular_energia_solar(
    hourly_data: dict,
    painel: dict,
    performance_ratio: float = 1.0,
    tilt_angle: float | None = None,
    degradacao_anual: float = 0.0,
    soiling_rate_per_hour: float = 0.005 / 24.0,
    soiling_cap: float = 0.20,
    rain_wash_efficiency: float = 1.0,
    rain_wash_mm_threshold: float = 5.0,
    module_azimuth_deg: Optional[float] = None,
    timezone_offset_hours: float = 0.0,
    logger=None,
):
    """
    Versão segura: assume radiações em W/m2 (já normalizadas).
    Principais mudanças:
     - Não tenta adivinhar unidades (remove duplicação).
     - logger defensivo.
     - renomeado coef > temp_coeff para evitar conflito com scipy.gamma.
     - soiling wash baseado em threshold (mm).
    """
    logger = logger or logging.getLogger(__name__)

    if not isinstance(hourly_data, dict) or "data" not in hourly_data:
        raise ValueError("hourly_data deve conter {'data': ..., 'units': ...}")

    df = pd.DataFrame(hourly_data["data"]).copy()
    if df.empty:
        raise ValueError("Dados horários vazios.")

    # dt_hours se fornecido (informativo)
    dt_hours = hourly_data.get("dt_hours", 1.0)

    # padroniza coluna/índice date
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.floor("h")
        df = df.dropna(subset=["date"]).reset_index(drop=True)
        df.index = pd.DatetimeIndex(df["date"].values)
    else:
        if isinstance(df.index, pd.DatetimeIndex):
            df.index = df.index.floor("h")
            df["date"] = df.index
        else:
            raise ValueError("Sem coluna 'date' e sem índice DatetimeIndex.")

    # parâmetros do painel
    potencia_kwp = float(painel.get("potencia_kwp", 0.0))
    p_stc_W = potencia_kwp * 1000.0
    n_paineis = int(painel.get("quantidade_paineis", 1))
    area_panel_m2 = float(painel.get("area_m2", 0.0))
    area_total_m2 = area_panel_m2 * n_paineis

    eta_ref = float(painel.get("eficiencia", np.nan))
    if not np.isfinite(eta_ref) or eta_ref <= 0:
        if area_panel_m2 > 0:
            eta_ref = p_stc_W / (1000.0 * area_panel_m2)
        else:
            raise ValueError("Eficiência referencial ausente e 'area_m2' inválida.")

    # coefficients: renomeado para evitar conflito com scipy.gamma
    coef_temp = float(painel.get("coef_temperatura", painel.get("coef_temperatura_voc", -0.0034)))
    U0 = float(painel.get("faiman_U0", 25.0))
    U1 = float(painel.get("faiman_U1", 6.84))
    b0 = float(painel.get("iam_b0", 0.05))

    # meteorologia básica: assume radiações já em W/m2
    Tamb = pd.to_numeric(df.get("temperature_2m", 25.0), errors="coerce").fillna(25.0)
    v10 = pd.to_numeric(df.get("wind_speed_10m", np.nan), errors="coerce")
    v100 = pd.to_numeric(df.get("wind_speed_100m", np.nan), errors="coerce")
    if v10.notna().sum() > 0:
        v_wind = v10.fillna(1.0)
    elif v100.notna().sum() > 0:
        v_wind = (v100 / ((100.0 / 10.0) ** 0.14)).fillna(1.0)
    else:
        v_wind = pd.Series(1.0, index=df.index)

    # --- ASSUME: GHI/DNI/DHI já em W/m2 (normalizado upstream) ---
    GHI_Wm2 = pd.to_numeric(df.get("shortwave_radiation", 0.0), errors="coerce").fillna(0.0).clip(lower=0.0).values
    DNI_Wm2 = pd.to_numeric(df.get("direct_radiation", 0.0), errors="coerce").fillna(0.0).clip(lower=0.0).values
    DHI_Wm2 = pd.to_numeric(df.get("diffuse_radiation", 0.0), errors="coerce").fillna(0.0).clip(lower=0.0).values

    # albedo
    alb = float(df.get("albedo", painel.get("albedo", 0.2)).iloc[0]) if "albedo" in df.columns else float(painel.get("albedo", 0.2))

    # geometria solar
    lat_deg = float(df.get("latitude", 0.0).iloc[0]) if "latitude" in df.columns else 0.0
    lon_deg = float(df.get("longitude", 0.0).iloc[0]) if "longitude" in df.columns else 0.0
    geom = _solar_geometry(df.index, lat_deg, lon_deg, timezone_offset_hours=timezone_offset_hours)
    cos_zenith = geom["cos_zenith"]
    solar_az = geom["azimuth_rad"]

    tilt_deg = float(painel.get("tilt_angle", tilt_angle if tilt_angle is not None else painel.get("tilt", 20.0)))
    tilt = radians(tilt_deg)
    module_az = 0.0 if module_azimuth_deg is None else radians(float(module_azimuth_deg))

    sin_z = np.sqrt(np.clip(1.0 - cos_zenith**2, 0.0, 1.0))
    cos_ti = np.clip(cos_zenith * np.cos(tilt) + sin_z * np.sin(tilt) * np.cos(solar_az - module_az), 0.0, 1.0)
    cos_zenith_safe = np.clip(cos_zenith, 1e-6, 1.0)

    with np.errstate(invalid="ignore", divide="ignore"):
        Ai = np.divide(DNI_Wm2, GHI_Wm2, out=np.zeros_like(DNI_Wm2), where=GHI_Wm2 > 0.0)
    Ai = np.clip(Ai, 0.0, 1.0)

    Hd_tilt = DHI_Wm2 * ((1.0 - Ai) * (1.0 + np.cos(tilt)) / 2.0 + Ai * (cos_ti / cos_zenith_safe))
    Hg_ref = GHI_Wm2 * alb * (1.0 - np.cos(tilt)) / 2.0
    Hb_tilt = DNI_Wm2 * cos_ti

    G_poa_Wm2 = np.clip(Hb_tilt + Hd_tilt + Hg_ref, 0.0, None)

    if "global_tilted_irradiance" in df.columns:
        GTI_arr = pd.to_numeric(df["global_tilted_irradiance"], errors="coerce").fillna(0.0).clip(lower=0.0).values
        mask_gti = GTI_arr > 0.0
        G_poa_Wm2[mask_gti] = GTI_arr[mask_gti]

    cos_ti_safe = np.clip(cos_ti, 1e-6, 1.0)
    IAM = np.clip(1.0 - b0 * (1.0 / cos_ti_safe - 1.0), 0.0, 1.0)

    # Faiman: usar temp_coeff
    v_arr = v_wind.values if hasattr(v_wind, "values") else np.full_like(G_poa_Wm2, 1.0)
    denom = (U0 + U1 * v_arr)
    denom = np.where(denom <= 0, U0, denom)
    with np.errstate(invalid="ignore", divide="ignore"):
        Tcell = Tamb.values + (G_poa_Wm2 * (1.0 - eta_ref)) / denom

    temp_ref = float(painel.get("temperatura_referencia", 25.0))
    temp_factor = 1.0 + coef_temp * (Tcell - temp_ref)
    temp_factor = np.clip(temp_factor, 0.4, 1.05)

    # soiling (com threshold explícito de chuva mm/h)
    rain = pd.to_numeric(df.get("rain", df.get("precipitation", 0.0)), errors="coerce").fillna(0.0).values
    soiling = np.zeros(len(df))
    acc = 0.0
    for i, r in enumerate(rain):
        if r > 0.0:
            # wash fraction proporcional ao precip, saturando em 1.0, e multiplicado por eficiência
            wash_frac = min(1.0, (r / max(rain_wash_mm_threshold, 1e-6)) ) * rain_wash_efficiency
            acc = acc * max(0.0, 1.0 - wash_frac)
        else:
            acc = min(soiling_cap, acc + soiling_rate_per_hour)
        soiling[i] = acc
    soiling = np.clip(soiling, 0.0, 1.0)

    # potência DC/AC etc...
    P_dc_W = G_poa_Wm2 * area_total_m2 * eta_ref * temp_factor * IAM * (1.0 - soiling)
    P_dc_kw = P_dc_W / 1000.0

    kWp_total_instalado = potencia_kwp * n_paineis
    dc_ac_ratio = float(painel.get("dc_ac_ratio", 1.2))
    ac_nominal_kw = float(painel.get("ac_power_kw", max(kWp_total_instalado  / max(dc_ac_ratio, 1e-6), 0.0)))
    if ac_nominal_kw <= 0:
        ac_nominal_kw = max(kWp_total_instalado  / max(dc_ac_ratio, 1e-6), 0.0)

    P_ac_kw = np.minimum(np.clip(P_dc_kw, 0.0, None), ac_nominal_kw)

    inverter_eff = float(painel.get("inverter_efficiency", 0.98))
    system_losses = float(painel.get("system_losses", 0.03))
    mismatch_loss = float(painel.get("mismatch_loss", 0.02))
    cable_loss = float(painel.get("cable_loss", 0.01))
    PR_losses = inverter_eff * (1.0 - system_losses) * (1.0 - mismatch_loss) * (1.0 - cable_loss)
    P_ac_kw *= float(performance_ratio) * PR_losses

    snow_mm = pd.to_numeric(df.get("snow_depth", 0.0), errors="coerce").fillna(0.0).values
    snow_block = np.clip(snow_mm / 50.0, 0.0, 1.0)
    P_ac_kw *= (1.0 - snow_block)

    df_out = pd.DataFrame(index=df.index)
    df_out["G_poa_Wm2"] = G_poa_Wm2
    df_out["cos_zenith"] = cos_zenith
    df_out["cos_incidence"] = cos_ti
    df_out["IAM"] = IAM
    df_out["Tcell_C"] = Tcell
    df_out["temp_factor"] = temp_factor
    df_out["soiling"] = soiling
    df_out["P_dc_kW"] = P_dc_kw
    df_out["P_ac_kW"] = P_ac_kw
    df_out["temperature_2m"] = Tamb.values
    df_out["wind_speed_10m"] = v_arr
    df_out["rain"] = rain
    df_out["energia_horaria_sistema_kwh"] = np.clip(P_ac_kw, 0.0, None)

    energia_diaria = df_out["energia_horaria_sistema_kwh"].groupby(df_out.index.date).sum()
    producao_anual = float(energia_diaria.sum())

    vida_util = int(max(1, int(painel.get("vida_util", 25))))
    if degradacao_anual <= 0.0:
        energia_vida = producao_anual * vida_util
    else:
        anos = np.arange(vida_util, dtype=float)
        energia_vida = float(np.sum(producao_anual * (1.0 - degradacao_anual) ** anos))

    ghi_annual_kwh_per_m2 = float(np.sum(G_poa_Wm2) / 1000.0)
    kWp_total_instalado = max(kWp_total_instalado, 1e-9)
    yield_spec = producao_anual / kWp_total_instalado
    area_per_kWp = (area_total_m2 / kWp_total_instalado) if kWp_total_instalado > 0 else 0.0

    if not np.isfinite(area_per_kWp) or area_per_kWp <= 0:
        typ_area_per_kWp = painel.get("area_m2_por_kwp", None)
        if typ_area_per_kWp:
            area_per_kWp = float(typ_area_per_kWp)
        else:
            area_per_kWp = np.clip(area_total_m2 / kWp_total_instalado, 0.5, 12.0)
    area_per_kWp = float(np.clip(area_per_kWp, 0.5, 12.0))

    yield_upper_physical = ghi_annual_kwh_per_m2 * eta_ref * area_per_kWp
    if yield_upper_physical > 0 and yield_spec > yield_upper_physical * 1.02:
        yield_spec = float(min(yield_spec, yield_upper_physical))

    capacidade_maxima = kWp_total_instalado * 8760.0
    fator_capacidade = float(np.clip(producao_anual / max(capacidade_maxima, 1e-9), 0.0, 1.0))

    return float(producao_anual), float(fator_capacidade), df_out, float(energia_vida)

# ============================================================
# EÓLICA
# ============================================================

#######################################################################
#  NOVO MÓDULO ROBUSTO PARA v_hub — USA v10 E v100 DE FORMA FÍSICA
#  - Sincroniza
#  - Filtra outliers
#  - Estima alpha via mediana móvel
#  - Combina v10 e v100 por pesos inversos da variância
#######################################################################
def compute_v_hub(df, hub_height, window_alpha=24, min_periods_alpha=6, sigma_min=0.1, alpha_max=0.25, logger=None):
    """
    Retorna DataFrame com v_hub. Mudanças:
     - rolling median sem center=True (causal).
     - logger defensivo.
    """
    logger = logger or logging.getLogger(__name__)

    df = df.sort_index()
    df = df[~df.index.duplicated(keep='first')]
    df = df.resample("1h").asfreq()

    v10 = df['wind_speed_10m'].astype(float).clip(0.0, 80.0)
    v100 = df['wind_speed_100m'].astype(float).clip(0.0, 80.0)

    v10 = v10.interpolate(limit=3)
    v100 = v100.interpolate(limit=3)

    eps = 1e-6
    alpha_raw = pd.Series(np.nan, index=df.index)
    mask_valid = (v10 > eps) & (v100 > eps)

    alpha_raw.loc[mask_valid] = np.log(v100.loc[mask_valid] / v10.loc[mask_valid]) / np.log(100.0/10.0)

    alpha_med = alpha_raw.rolling(
        window=window_alpha,
        min_periods=min_periods_alpha,
        center=False  # causal, não usa futuros
    ).median()

    alpha = alpha_med.ffill().bfill().clip(0.03, alpha_max)

    sigma10 = v10.rolling(window=window_alpha, min_periods=6).std().fillna(sigma_min)
    sigma100 = v100.rolling(window=window_alpha, min_periods=6).std().fillna(sigma_min)
    sigma10 = sigma10.clip(lower=sigma_min)
    sigma100 = sigma100.clip(lower=sigma_min)

    v_hub_10 = v10 * (hub_height / 10.0) ** alpha
    v_hub_100 = v100 * (hub_height / 100.0) ** alpha

    w10 = 1.0 / (sigma10 ** 2)
    w100 = 1.0 / (sigma100 ** 2)
    wsum = w10 + w100
    w10 = w10 / wsum
    w100 = w100 / wsum

    v_hub = w10 * v_hub_10 + w100 * v_hub_100

    v_hub = v_hub.where(mask_valid | (v10 > eps), v10 * (hub_height / 10.0) ** 0.14)
    v_hub = v_hub.clip(0.0, 80.0)

    out = pd.DataFrame({
        'v10': v10,
        'v100': v100,
        'alpha_raw': alpha_raw,
        'alpha': alpha,
        'sigma10': sigma10,
        'sigma100': sigma100,
        'v_hub_10': v_hub_10,
        'v_hub_100': v_hub_100,
        'v_hub': v_hub
    })

    return out

def calcular_energia_eolica(
    hourly_data,
    turbina: dict,
    k_forma: float | None = None,
    considerar_hub_height: bool = True,
    alpha_shear: float = 0.14,
    perdas_sistema: float = 0.05,
    disponibilidade: float = 0.98,
    densidade_padrao: float = 1.225,
    min_amostras_empirico: int = 7000,
    wake_loss_frac: float = 0.0,
    power_curve_table: list | None = None,
    max_integration_v: float | None = None,
    alpha_max: float = 0.25,
    logger=None,
):
    """
    Versão refinada de calcular_energia_eolica:
     - detecção robusta de unidade de pressão (hPa vs Pa)
     - escala de densidade aplicada corretamente (power ∝ rho)
     - alinhamento entre arrays (densidade vs velocidades)
     - tratamento seguro de casos com poucos dados
    """

    logger = logger or logging.getLogger(__name__)

    if isinstance(hourly_data, dict):
        df = pd.DataFrame(hourly_data.get("data", [])).copy()
    else:
        df = pd.DataFrame(hourly_data).copy()
    if df.empty:
        raise ValueError("Dados horários vazios.")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"]).reset_index(drop=True)
        df.index = pd.DatetimeIndex(df["date"].values)

    v10 = pd.to_numeric(df.get("wind_speed_10m", np.nan), errors="coerce")
    v100 = pd.to_numeric(df.get("wind_speed_100m", np.nan), errors="coerce")
    altura_ref = float(turbina.get("altura_ref", 10.0))
    hub_height = float(turbina.get("hub_height", altura_ref)) if considerar_hub_height else altura_ref

    ##################################################################
    #  NOVO CÁLCULO: shear robusto usando v10 e v100
    ##################################################################
    hub_height = float(turbina.get("hub_height", altura_ref))

    vprofile = compute_v_hub(df, hub_height=hub_height, alpha_max=alpha_max)

    df["v_hub"] = vprofile["v_hub"]
    df["v10"] = vprofile["v10"]
    df["v100"] = vprofile["v100"]

    # Debug completo do perfil de vento
    try:
        print("\n[DEBUG EOLICA] v10/v100/v_hub.describe():")
        print(df[['v10','v100','v_hub']].describe())
    except Exception as e:
        print("[DEBUG EOLICA] erro ao imprimir describe:", e)



    # densidade do ar: detectar unidade de pressure (hPa vs Pa)
    if "surface_pressure" in df.columns and "temperature_2m" in df.columns:
        p_raw = pd.to_numeric(df["surface_pressure"], errors="coerce")
        med_p = float(np.nanmedian(p_raw.values)) if p_raw.notna().sum() > 0 else 1013.0
        if med_p > 2000:  # Pa
            p_pa = p_raw.fillna(101325.0)
        else:  # hPa -> Pa
            p_pa = p_raw.fillna(1013.0) * 100.0
        T_k = pd.to_numeric(df["temperature_2m"], errors="coerce").fillna(20.0) + 273.15
        with np.errstate(invalid="ignore", divide="ignore"):
            rho_series = p_pa / (287.05 * T_k)
        rho_series = rho_series.clip(lower=0.5, upper=1.6).fillna(densidade_padrao)
        rho = rho_series
    else:
        rho = pd.Series(densidade_padrao, index=df.index)

    # garantir série de densidade alinhada ao índice horário (reindex + fill)
    try:
        rho_series = pd.Series(rho).astype(float)
        rho_series.index = df.index[:len(rho_series)] if len(rho_series) != len(df.index) else df.index
        # reindex explicitamente ao índice horário do df e preencher com valor padrão
        rho_aligned = rho_series.reindex(df.index).fillna(densidade_padrao).clip(lower=0.5, upper=1.6)
        dens_scale = (rho_aligned / densidade_padrao).astype(float).values
    except Exception:
        # fallback conservador: use densidade padrão
        logger.warning("[DENS] Falha ao alinhar rho -> usando densidade padrão para todo o período")
        dens_scale = np.full(len(df.index), float(densidade_padrao / densidade_padrao))



    # curva de potência
    pot_nominal_kw = float(turbina.get("potencia_nominal", 0.0))
    if pot_nominal_kw <= 0:
        raise ValueError("potencia_nominal inválida na turbina (deve estar em kW).")

    n_turb = int(turbina.get("quantidade_turbinas", 1))
    cut_in = float(turbina.get("cut_in_speed", 3.0))
    rated = float(turbina.get("rated_speed", turbina.get("rated_speed", 12.0)))
    cut_out = float(turbina.get("cut_out_speed", 25.0))
    if not (0 < cut_in < rated < cut_out):
        raise ValueError("Curva de velocidades inválida (cut_in < rated < cut_out exigidos).")

    def pc_default(v):
        v = np.asarray(v, float)
        p = np.zeros_like(v, dtype=float)
        m1 = (v >= cut_in) & (v <= rated)
        m2 = (v > rated) & (v <= cut_out)
        p[m1] = pot_nominal_kw * ((v[m1] - cut_in) / max((rated - cut_in), 1e-6)) ** 3
        p[m2] = pot_nominal_kw
        return p

    if power_curve_table:
        table = sorted(power_curve_table, key=lambda x: float(x[0]))
        vs = np.array([float(x[0]) for x in table])
        ps = np.array([float(x[1]) for x in table])
        def pc_interp(v): return np.interp(v, vs, ps, left=0.0, right=ps[-1])
        pc = pc_interp
    else:
        pc = pc_default

    fator_oper = (1.0 - perdas_sistema) * disponibilidade * (1.0 - wake_loss_frac)
    HORAS_ANO = 8760.0

    usar_empirico = len(df) >= min_amostras_empirico
    if usar_empirico:
        v_arr = df["v_hub"].values
        P_kw_unidade = pc(v_arr) * dens_scale * fator_oper
        energia_kwh_total = float(np.nansum(P_kw_unidade * n_turb))
        df["P_kw_corrigido"] = P_kw_unidade
        df["energia_horaria_kwh_total"] = P_kw_unidade * n_turb
    else:
        v = df["v_hub"].dropna().values
        if v.size < 50:
            raise ValueError("Dados insuficientes para estimativa teórica (menos de 50 amostras).")
        v_mean = v.mean()
        v_std = v.std(ddof=1) if v.size > 1 else 0.0
        R = v_std / v_mean if v_mean > 0 else 0.0

        def solve_k_from_R(R_local):
            if R_local <= 0 or not np.isfinite(R_local):
                return 2.0
            def f(k): return np.sqrt(gamma(1+2.0/k)/gamma(1+1.0/k)**2 - 1.0) - R_local
            try:
                k_sol = brentq(f, 0.5, 20.0, maxiter=200)
                return float(k_sol)
            except Exception:
                return max(0.5, R_local ** -1.086)

        k = solve_k_from_R(R) if k_forma is None else float(k_forma)
        c = v_mean / gamma(1.0 + 1.0 / k)
        dens_mean = float(np.nanmean(dens_scale))
        def pdf_weibull(vv):
            vv = np.asarray(vv, float)
            vv = np.maximum(vv, 0.0)
            return (k / c) * (vv / c) ** (k - 1.0) * np.exp(- (vv / c) ** k)

        vmax = max_integration_v or cut_out
        def integrand(vv):
            return pc([vv])[0] * dens_mean * pdf_weibull(vv)

        try:
            energia_kw, _ = quad(integrand, 0.0, vmax, epsabs=1e-6, epsrel=1e-6, limit=200)
        except Exception:
            xs = np.linspace(0.0, vmax, 5000)
            energia_kw = np.trapz(pc(xs) * dens_mean * pdf_weibull(xs), xs)

        energia_kwh_total = float(energia_kw * HORAS_ANO * n_turb * fator_oper)
        df["P_kw_corrigido"] = np.nan
        df["energia_horaria_kwh_total"] = np.nan

    # --- DEBUG APÓS A CRIAÇÃO DAS COLUNAS (LOCAL CORRETO) ---
    try:
        print("\n[DEBUG EOLICA] P_kw_corrigido.describe():")
        print(df[['P_kw_corrigido']].describe())

        print("\n[DEBUG EOLICA] energia_horaria_kwh_total.describe():")
        print(df[['energia_horaria_kwh_total']].describe())
    except Exception as e:
        print("[DEBUG EOLICA] ERRO describe:", e)


    vida = int(turbina.get("vida_util", 25))
    degrad = float(turbina.get("degradacao_anual", 0.005))
    if degrad <= 0.0:
        energia_vida = energia_kwh_total * vida
    else:
        anos = np.arange(vida, dtype=float)
        energia_vida = float(np.sum(energia_kwh_total * (1.0 - degrad) ** anos))

    cap_max = pot_nominal_kw * n_turb * HORAS_ANO
    fc = float(np.clip(energia_kwh_total / max(cap_max, 1e-9), 0.0, 1.0))

    # --- GARANTIA ANTES DO return (IMPRESCINDÍVEL) ---
    for col in ["P_kw_corrigido", "energia_horaria_kwh_total"]:
        if col not in df.columns:
            df[col] = np.nan


    df_out = df[["v_hub", "P_kw_corrigido", "energia_horaria_kwh_total"]].copy()
    return float(energia_kwh_total), float(fc), df_out, float(energia_vida)

###💰 BLOCO 4 — Modelos econômico-energéticos integrados

In [ ]:
# ============================================================
# 💰 BLOCO 3 — Modelos econômico-energéticos integrados
# ============================================================

import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Optional

# ---------------------------
# UTILS FINANCEIROS/BÁSICOS
# ---------------------------
def _crf(rate: float, n: int) -> float:
    """Capital recovery factor. rate decimal (ex: 0.08), n anos."""
    rate = float(rate)
    n = int(n)
    if n <= 0:
        return 1.0
    if abs(rate) < 1e-12:
        return 1.0 / n
    return (rate * (1 + rate) ** n) / ((1 + rate) ** n - 1)

def _pv_factor(rate: float, t: int) -> float:
    """PV factor for payment at end of period t (t integer >=1)."""
    return 1.0 / ((1.0 + rate) ** t)

# ---------------------------
# LCOE
# ---------------------------
def calcular_lcoe(
    producoes_anuais: dict,
    capex_total: float,
    vida_util: int,
    tarifa_r_kwh: float,
    degradacao_anual: float = 0.005,
    om_percentual_capex: float = 0.01,
    taxa_desconto: float = 0.08,
    logger=None,
):
    """
    LCOE econômico consistente: considera receita, opex, degradação e desconto.
    SE LCOE < tarifa => NPV > 0 obrigatoriamente (verificação feita).
    Retorna dict com LCOE, NPV e parcelas PV.
    """
    from math import isfinite

    if logger is None:
        import logging
        logger = logging.getLogger("energy_analysis")

    logger.info("⚙️ [LCOE_ECON] INÍCIO calcular_lcoe_economico")
    logger.debug(f"[LCOE_ECON] capex_total={capex_total:,.2f} | vida_util={vida_util} | tarifa={tarifa_r_kwh}")

    anos_ordenados = sorted(producoes_anuais.keys())
    if len(anos_ordenados) == 0:
        raise ValueError("producoes_anuais vazio")

    pv_receita = 0.0
    pv_energia = 0.0
    pv_opex = 0.0

    for i, ano in enumerate(anos_ordenados):
        energia = float(producoes_anuais[ano]) * ((1.0 - float(degradacao_anual)) ** i)
        receita = energia * float(tarifa_r_kwh)
        opex = capex_total * float(om_percentual_capex)

        fator_desc = 1.0 / ((1.0 + float(taxa_desconto)) ** (i + 1))

        pv_receita += receita * fator_desc
        pv_energia += energia * fator_desc
        pv_opex += opex * fator_desc

        logger.trace if hasattr(logger, "trace") else logger.debug
        logger.debug(
            f"[LCOE_ECON] ano={ano} | energia={energia:,.2f} kWh | receita={receita:,.2f} | "
            f"opex={opex:,.2f} | pv_factor={fator_desc:.6f}"
        )

    npv_projeto = pv_receita - (capex_total + pv_opex)

    if pv_energia <= 0 or not isfinite(pv_energia):
        raise ValueError("PV energia inválida — verifique producoes_anuais/degradacao.")

    # LCOE econômico: custo presente por kWh produzido presente
    lcoe_econ = (capex_total + pv_opex) / pv_energia

    logger.info(f"[LCOE_ECON] PV_receita={pv_receita:,.2f} | PV_opex={pv_opex:,.2f} | PV_energia={pv_energia:,.2f}")
    logger.info(f"[LCOE_ECON] LCOE={lcoe_econ:.6f} R$/kWh | NPV={npv_projeto:,.2f}")

    # Consistência financeira: LCOE < tarifa <=> npv_projeto > 0
    inconsistent = False
    if (lcoe_econ < tarifa_r_kwh and npv_projeto < -1e-6) or (lcoe_econ > tarifa_r_kwh and npv_projeto > 1e-6):
        inconsistent = True
        logger.error("❌ INCONSISTÊNCIA FINANCEIRA DETECTADA — LCOE e NPV não batem com a tarifa informada")
        logger.error(f"   LCOE={lcoe_econ:.6f} vs Tarifa={tarifa_r_kwh} | NPV={npv_projeto:,.2f}")

    if not inconsistent:
        logger.info("✔️ Consistência financeira OK — LCOE ↔ NPV alinhados")

    return {
        "LCOE": float(lcoe_econ),
        "NPV": float(npv_projeto),
        "PV_receita": float(pv_receita),
        "PV_opex": float(pv_opex),
        "PV_energia": float(pv_energia),
    }

# ---------------------------
# EROI
# ---------------------------
def calcular_eroi(
    potencia_kw: float,
    vida_util: int,
    producao_anual_kwh: Optional[float] = None,
    fator_capacidade: Optional[float] = None,
    fonte: str = "generica",
    energia_investida_kwh: Optional[float] = None,
    energia_embutida_kwh_per_kw_base: Optional[Dict[str, float]] = None,
    alpha_scale: float = 0.95,
    incluir_transmissao: bool = True,
    distancia_transmissao_km: float = 200.0,
    energia_tx_kwh_per_kw_per_km: Optional[float] = None,
    perdas_operacionais_frac: Optional[Dict[str, float]] = None,
    eficiencia_conversao: float = 0.98,
    infra_extra_kwh_per_kw: float = 0.0,
    degradacao_anual: float = 0.005,
    return_details: bool = False
) -> Dict:
    """
    Calcula EROI = Energia_gerada_total / Energia_investida_total
    energia_embutida_kwh_per_kw_base: dicionário de valores por tecnologia
    alpha_scale: expoente de escala para ganhos de escala (<=1)
    energia_tx_kwh_per_kw_per_km: energia para transmissao por kW por km (fallback)
    perdas_operacionais_frac: frações operacionais removidas da energia gerada (ex: eficiencia de disponibilidade)
    """
    if vida_util <= 0 or potencia_kw <= 0:
        raise ValueError("vida_util e potencia_kw devem ser positivos.")

    # defaults baseados em literatura aproximada (valores indicativos)
    energia_embutida_kwh_per_kw_base = energia_embutida_kwh_per_kw_base or {
        "solar": 2500.0, "eolica": 600.0, "hidreletrica": 3500.0,
        "termica": 1200.0, "bateria": 8000.0, "generica": 2000.0,
    }

    perdas_operacionais_frac = perdas_operacionais_frac or {
        "eolica": 0.045, "solar": 0.025, "termica": 0.08,
        "hidreletrica": 0.03, "bateria": 0.10, "generica": 0.04,
    }

    perda_frac = perdas_operacionais_frac.get(fonte.lower(), perdas_operacionais_frac["generica"])

    # energia embutida total (kWh) por potência instalada (kW)
    base = energia_embutida_kwh_per_kw_base.get(fonte.lower(), energia_embutida_kwh_per_kw_base["generica"])
    if energia_investida_kwh is None:
        energia_investida_kwh = (base + infra_extra_kwh_per_kw) * (potencia_kw ** alpha_scale)
        if incluir_transmissao:
            energia_tx_kwh_per_kw_per_km = energia_tx_kwh_per_kw_per_km or 1.5
            energia_investida_kwh += energia_tx_kwh_per_kw_per_km * distancia_transmissao_km * potencia_kw

    if energia_investida_kwh <= 0:
        raise ValueError("Energia investida inválida (≤0).")

    # energia gerada ao longo da vida (kWh)
    energia_total_gerada = 0.0
    horas_ano = 8760
    if producao_anual_kwh is not None:
        for ano in range(vida_util):
            energia_total_gerada += producao_anual_kwh * ((1.0 - degradacao_anual) ** ano) * (1.0 - perda_frac) * eficiencia_conversao
    elif fator_capacidade is not None:
        for ano in range(vida_util):
            energia_total_gerada += potencia_kw * fator_capacidade * horas_ano * ((1.0 - degradacao_anual) ** ano) * (1.0 - perda_frac) * eficiencia_conversao
    else:
        raise ValueError("Forneça producao_anual_kwh ou fator_capacidade para calcular produção.")

    eroi_val = energia_total_gerada / energia_investida_kwh

    result = {
        "EROI": float(eroi_val),
        "Energia_gerada_total_kWh": float(energia_total_gerada),
        "Energia_investida_total_kWh": float(energia_investida_kwh),
        "Fonte": fonte,
        "Perda_operacional_frac": float(perda_frac),
        "Eficiencia_conversao_frac": float(eficiencia_conversao),
        "Alpha_escala": float(alpha_scale),
    }

    if return_details:
        result["Parametros"] = {
            "potencia_kw": potencia_kw,
            "vida_util": vida_util,
            "degradacao_anual": degradacao_anual
        }

    return result

# ---------------------------
# ENERGIA INVESTIDA (detalhada)
# ---------------------------
def calcular_energia_investida(
    sistema: Dict,
    tipo: str,
    energia_por_kw: Optional[float] = None,
    incluir_om_energetico: bool = False,
    om_energetico_anual_por_kw: Optional[float] = None,
    vida_util: Optional[int] = None,
    incluir_transporte: bool = True,
    distancia_transporte_km: float = 200.0,
    transporte_kwh_per_kw_per_km: Optional[float] = None,
    incluir_transmissao: bool = True,
    distancia_transmissao_km: float = 200.0,
    transmissao_kwh_por_kw_per_km: Optional[float] = None,
    alpha_escala: float = 0.95,
    eficiencia_fabrica: float = 1.0,
    taxa_aprendizado_om: float = 0.02,
    taxa_reciclagem: float = 0.0,
    infra_extra_kwh_per_kw: float = 0.0,
    preco_energia_kwh: Optional[float] = None,
    taxa_desconto: float = 0.08
) -> Dict:
    """
    Calcula energia embutida total (fabricação + transporte + transmissão + O&M energético - reciclagem).
    `sistema` deve conter parâmetros como:
      - solar: potencia_kwp (kW por painel), quantidade_paineis
      - eolica: potencia_nominal (kW), quantidade_turbinas
    Retorna dict com decomposição.
    """
    tipo = tipo.lower()
    if tipo == "solar":
        potencia_total_kw = float(sistema.get("potencia_kwp", 0.0)) * int(sistema.get("quantidade_paineis", 0))
        energia_por_kw = energia_por_kw or 2500.0
        transporte_kwh_per_kw_per_km = transporte_kwh_per_kw_per_km or 1.5
        transmissao_kwh_por_kw_per_km = transmissao_kwh_por_kw_per_km or 1.5
    elif tipo == "eolica":
        potencia_total_kw = float(sistema.get("potencia_nominal", 0.0)) * int(sistema.get("quantidade_turbinas", 0))
        energia_por_kw = energia_por_kw or 900.0
        transporte_kwh_per_kw_per_km = transporte_kwh_per_kw_per_km or 2.0
        transmissao_kwh_por_kw_per_km = transmissao_kwh_por_kw_per_km or 2.0
    else:
        raise ValueError("tipo deve ser 'solar' ou 'eolica'")

    if potencia_total_kw <= 0:
        raise ValueError("potencia_total_kw inválida (≤0).")

    # fabricação com escala: E_fab = energia_por_kw * potência_total_kw^(alpha_escala)
    energia_fabricacao_total_kwh = energia_por_kw * (potencia_total_kw ** alpha_escala) * eficiencia_fabrica

    # transporte
    transporte_energy_kwh = 0.0
    if incluir_transporte:
        transporte_energy_kwh = transporte_kwh_per_kw_per_km * distancia_transporte_km * (potencia_total_kw ** alpha_escala)

    # transmissao (energia por kW por km * potência_total * distancia)
    transmissao_energy_kwh = 0.0
    if incluir_transmissao:
        transmissao_energy_kwh = transmissao_kwh_por_kw_per_km * distancia_transmissao_km * potencia_total_kw

    # O&M energético
    energia_om_total_kwh = 0.0
    if incluir_om_energetico:
        if vida_util is None:
            raise ValueError("vida_util necessário para incluir O&M energético")
        om_energetico_anual_por_kw = om_energetico_anual_por_kw or (150.0 if tipo == "solar" else 250.0)
        for ano in range(vida_util):
            fator_apr = (1.0 - taxa_aprendizado_om) ** ano
            energia_om_total_kwh += om_energetico_anual_por_kw * potencia_total_kw * fator_apr

    # reciclagem
    energia_reciclada_kwh = 0.0
    if taxa_reciclagem > 0.0:
        recuperavel = energia_fabricacao_total_kwh + transporte_energy_kwh + transmissao_energy_kwh
        energia_reciclada_kwh = min(taxa_reciclagem * recuperavel, recuperavel)

    energia_total_bruta_kwh = energia_fabricacao_total_kwh + transporte_energy_kwh + transmissao_energy_kwh + energia_om_total_kwh
    energia_total_liquida_kwh = max(0.0, energia_total_bruta_kwh - energia_reciclada_kwh)

    custo_energetico_total = None
    if preco_energia_kwh is not None and vida_util is not None:
        # distribuir custo energético ao longo da vida (NPV usando taxa_desconto)
        fluxo = []
        for ano in range(vida_util):
            fluxo_ano = energia_total_bruta_kwh / vida_util
            fluxo.append(fluxo_ano / ((1 + taxa_desconto) ** (ano + 1)))
        custo_energetico_total = sum(fluxo) * preco_energia_kwh

    return {
        "energia_fabricacao_total_kwh": float(energia_fabricacao_total_kwh),
        "energia_transporte_kwh": float(transporte_energy_kwh),
        "energia_transmissao_kwh": float(transmissao_energy_kwh),
        "energia_om_total_kwh": float(energia_om_total_kwh),
        "energia_reciclada_kwh": float(energia_reciclada_kwh),
        "energia_total_bruta_kwh": float(energia_total_bruta_kwh),
        "energia_total_liquida_kwh": float(energia_total_liquida_kwh),
        "potencia_total_kw": float(potencia_total_kw),
        "custo_energetico_total": float(custo_energetico_total) if custo_energetico_total is not None else None,
    }

# ---------------------------
# PAYBACK (nominal e descontado)
# ---------------------------
def calcular_payback(
    capex_total: float,
    producao_anual_kwh: float,
    tarifa_r_kwh: float = 0.70,
    vida_util: int = 25,
    degradacao_anual: float = 0.005,
    om_fixo_anual: float = 0.0,
    om_percentual: float = 0.01,
    taxa_desconto: float = 0.08,
    replacement_costs: Optional[List[Tuple[int, float]]] = None,
    salvage_value: float = 0.0,
    escalonamento_tarifa_anual: float = 0.0,
    escalonamento_om_anual: float = 0.0,
    return_cashflow: bool = False
) -> Dict:
    """
    Calcula payback nominal e descontado por interpolação linear entre anos.
    Retorna dict com payback_nominal_anos, payback_descontado_anos, e arrays de fluxos.
    """
    if vida_util < 1:
        raise ValueError("vida_util deve ser >=1")
    if capex_total < 0 or producao_anual_kwh < 0:
        raise ValueError("capex_total e producao_anual_kwh devem ser >=0")

    replacement_costs = replacement_costs or []
    n = int(vida_util)
    fluxo = np.zeros(n + 1, dtype=float)
    fluxo[0] = -float(capex_total)

    om_percentual = float(np.clip(om_percentual, 0.0, 1.0))
    om_base_percentual_val = om_percentual * float(capex_total)

    for t in range(1, n + 1):
        # produção com degradação
        prod_t = producao_anual_kwh * ((1.0 - degradacao_anual) ** (t - 1))
        tarifa_t = tarifa_r_kwh * ((1 + escalonamento_tarifa_anual) ** (t - 1))
        receita_t = prod_t * tarifa_t

        # O&M
        om_fixo_t = om_fixo_anual * ((1 + escalonamento_om_anual) ** (t - 1))
        om_t = om_fixo_t + om_base_percentual_val

        # substituições
        subs_t = sum(v for ano_rep, v in replacement_costs if int(ano_rep) == t)

        fluxo[t] = receita_t - (om_t + subs_t)

    # salvage no último ano
    if salvage_value:
        fluxo[n] += float(salvage_value)

    # descontar fluxos
    if taxa_desconto == 0:
        fluxo_descontado = fluxo.copy()
    else:
        fatores = np.array([1.0 / ((1.0 + taxa_desconto) ** t) for t in range(n + 1)], dtype=float)
        fluxo_descontado = fluxo * fatores

    acum_nom = np.cumsum(fluxo)
    acum_desc = np.cumsum(fluxo_descontado)

    def interp_pb(acum: np.ndarray) -> float:
        pos = np.where(acum >= 0)[0]
        if len(pos) == 0:
            return float('inf')
        i = pos[0]
        if i == 0:
            return 0.0
        prev, curr = float(acum[i - 1]), float(acum[i])
        if curr == prev:
            return float(i)
        return float((i - 1) + (-prev) / (curr - prev))

    pb_nom = interp_pb(acum_nom)
    pb_desc = interp_pb(acum_desc)

    out = {
        "payback_nominal_anos": float(pb_nom),
        "payback_descontado_anos": float(pb_desc),
        "fluxo_nominal": fluxo,
        "fluxo_descontado": fluxo_descontado,
        "acumulado_nominal": acum_nom,
        "acumulado_descontado": acum_desc,
        "parametros": {
            "CAPEX_total": float(capex_total),
            "tarifa_r_kwh": float(tarifa_r_kwh),
            "taxa_desconto": float(taxa_desconto),
            "vida_util": int(vida_util),
            "OM_fixo_anual": float(om_fixo_anual),
            "OM_percentual_CAPEX": float(om_percentual),
            "salvage_value": float(salvage_value)
        }
    }

    if return_cashflow:
        out["cashflow"] = fluxo

    return out


### 📌BLOCO 5 — Execução final

In [ ]:
import matplotlib.pyplot as plt

# ============================================================
# PRODUÇÃO ANUAL E FATOR DE CAPACIDADE (solar/eólica)
# ============================================================
def calcular_producao_e_fator_capacidade_anual(
    hourly_data,
    tipo_energia,
    painel: dict | None = None,
    turbina: dict | None = None,
    prefer_normalizar_ano_parcial: bool = False,
    min_frac_hours_for_full_year: float = 0.95,
    max_normalization_factor: float = 1.10,
    perdas_sistema_extra: float = 0.0,
    devolver_detalhes: bool = False
):
    """
    Versão revisada com validação temporal rigorosa e passagem de parâmetros relevantes
    para calcular_energia_solar/calcular_energia_eolica. Mantém assinatura antiga.
    """
    df = pd.DataFrame(hourly_data["data"] if isinstance(hourly_data, dict) else hourly_data).copy()
    units = hourly_data.get("units", {}) if isinstance(hourly_data, dict) else {}

    # blindagem temporal forte
    df = ensure_no_date_ambiguity(df)
    if "date" not in df.columns:
        raise ValueError("Coluna 'date' ausente nos dados meteorológicos.")

    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.floor("h")
    df.dropna(subset=["date"], inplace=True)
    df.sort_values("date", inplace=True)
    df["ano"] = df["date"].dt.year
    df.reset_index(drop=True, inplace=True)
    df.index.name = None

    tipo = str(tipo_energia).lower().strip()
    if tipo not in ("solar", "eolica"):
        raise ValueError("tipo_energia deve ser 'solar' ou 'eolica'")

    HORAS_ANO = 8760
    producoes_anuais, fatores_capacidade_anual, horas_por_ano, detalhes = {}, {}, {}, {}

    # defaults de passagem para calcular_energia_solar
    timezone_offset_hours = float(painel.get("timezone_offset_hours", 0.0)) if painel else 0.0
    module_azimuth_deg = painel.get("module_azimuth_deg", None) if painel else None
    performance_ratio_from_panel = painel.get("performance_ratio", 1.0) if painel else 1.0

    for ano, g in df.groupby("ano", sort=True):
        g = ensure_no_date_ambiguity(g.copy())
        n_horas = len(g)
        if n_horas == 0:
            continue

        frac_horas = n_horas / HORAS_ANO
        if frac_horas < min_frac_hours_for_full_year and not prefer_normalizar_ano_parcial:
            # pular ano insuficiente se preferência não permitir normalização
            continue

        # fator de normalização com teto físico
        fator_norm_raw = HORAS_ANO / max(n_horas, 1)
        fator_norm = min(fator_norm_raw, max_normalization_factor)
        if fator_norm_raw > max_normalization_factor:
            logger.warning(f"[NORMALIZAÇÃO] Ano {ano}: fator_norm bruto={fator_norm_raw:.3f} limitado a {max_normalization_factor:.3f}")

        dados_ano = {"data": g.reset_index(drop=True), "units": units}

        if tipo == "solar":
            # Passa timezone/module_az/performance_ratio para a função solar (assume ela faz validações internas)
            prod, fc, df_hora, _ = calcular_energia_solar(
                dados_ano,
                painel or {},
                performance_ratio=float(performance_ratio_from_panel),
                tilt_angle=(painel.get("tilt_angle") if painel else None),
                degradacao_anual=float(painel.get("degradacao_anual", 0.0)) if painel else 0.0,
                soiling_rate_per_hour=float(painel.get("soiling_rate_per_hour", 0.005 / 24.0)) if painel else 0.005 / 24.0,
                soiling_cap=float(painel.get("soiling_cap", 0.20)) if painel else 0.20,
                rain_wash_efficiency=float(painel.get("rain_wash_efficiency", 1.0)) if painel else 1.0,
                module_azimuth_deg=module_azimuth_deg,
                timezone_offset_hours=timezone_offset_hours,
            )

        else:
            prod, fc, df_hora, _ = calcular_energia_eolica(
                dados_ano,
                turbina or {},
                k_forma=(turbina.get("k_forma") if turbina else None),
                considerar_hub_height=True,
                alpha_shear=(turbina.get("alpha_shear", 0.14) if turbina else 0.14),
                perdas_sistema=(turbina.get("perdas_sistema", 0.05) if turbina else 0.05),
                disponibilidade=(turbina.get("disponibilidade", 0.98) if turbina else 0.98),
            )


        # aplica normalização (se desejado/permitido) e perdas extras
        prod_ajust = float(prod) * fator_norm * (1.0 - perdas_sistema_extra)
        pot_kw = (
            (painel.get("potencia_kwp", 0.0) * painel.get("quantidade_paineis", 0))
            if tipo == "solar"
            else (turbina.get("potencia_nominal", 0.0) * turbina.get("quantidade_turbinas", 0))
        )
        energia_max = pot_kw * HORAS_ANO
        fc_ajust = np.clip(prod_ajust / max(energia_max, 1e-9), 0.0, 1.0)

        producoes_anuais[int(ano)] = prod_ajust
        fatores_capacidade_anual[int(ano)] = float(fc_ajust)
        horas_por_ano[int(ano)] = n_horas

        if devolver_detalhes:
            detalhes[int(ano)] = {
                "df_hora": df_hora,
                "producao_modelo_kWh": float(prod),
                "producao_ajustada_kWh": float(prod_ajust),
                "fator_capacidade_modelo": float(fc),
                "fator_capacidade_ajustado": float(fc_ajust),
                "n_horas": n_horas,
                "frac_horas_ano": frac_horas,
                "fator_normalizacao": float(fator_norm),
            }

    if not producoes_anuais:
        return ({}, {}, 0.0, 0.0, {}) if devolver_detalhes else ({}, {}, 0.0, 0.0)

    anos = np.array(sorted(producoes_anuais.keys()), dtype=int)
    coberturas = np.array([horas_por_ano[a] for a in anos], dtype=float)
    pesos = coberturas / coberturas.sum()
    prod_vals = np.array([producoes_anuais[a] for a in anos], dtype=float)
    fc_vals = np.array([fatores_capacidade_anual[a] for a in anos], dtype=float)

    media_producao = float(np.dot(prod_vals, pesos))
    media_fc = float(np.dot(fc_vals, pesos))

    return (producoes_anuais, fatores_capacidade_anual, media_producao, media_fc, detalhes) \
        if devolver_detalhes else \
        (producoes_anuais, fatores_capacidade_anual, media_producao, media_fc)


In [ ]:
def gerar_graficos(
    producao_solar_anual=None,
    producao_eolica_anual=None,
    fatores_capacidade_solar=None,
    fatores_capacidade_eolica=None,
    lcoe_solar_rpkwh=None,
    lcoe_eolica_rpkwh=None,
    eroi_solar=None,
    eroi_eolica=None,
    lcoe_solar_details=None,
    lcoe_eolica_details=None,
    investimentos=None,
    df=None,
    payback_solar=None,
    payback_eolica=None,
    tarifa_r_kwh: float = 0.7,
    horizonte_anos: int = 20,
):
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    plt.style.use("ggplot")

    cores = {
        "solar": "#FFA500",
        "eolica": "#0057B8",
        "rede_const": "#555555",
        "rede_1": "#0077BB",
        "rede_3": "#00AA55",
        "rede_5": "#DD3333",
    }

    roi_resultados = {}

    # =====================================================
    # 1️⃣ PRODUÇÃO ANUAL
    # =====================================================
    if producao_solar_anual and producao_eolica_anual:
        anos = sorted(producao_solar_anual.keys())
        plt.figure(figsize=(10, 5))
        plt.plot(anos, [producao_solar_anual[a] for a in anos],
                 label="Solar (kWh/ano)", color=cores["solar"], marker="o")
        plt.plot(anos, [producao_eolica_anual[a] for a in anos],
                 label="Eólica (kWh/ano)", color=cores["eolica"], marker="s")
        plt.title("Produção anual — Solar vs Eólica")
        plt.xlabel("Ano")
        plt.ylabel("Energia [kWh]")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    # =====================================================
    # 2️⃣ FATOR DE CAPACIDADE
    # =====================================================
    if fatores_capacidade_solar and fatores_capacidade_eolica:
        anos = sorted(fatores_capacidade_solar.keys())
        plt.figure(figsize=(10, 5))
        plt.plot(anos, [fatores_capacidade_solar[a] for a in anos],
                 label="Solar", color=cores["solar"], marker="o")
        plt.plot(anos, [fatores_capacidade_eolica[a] for a in anos],
                 label="Eólica", color=cores["eolica"], marker="s")
        plt.title("Fator de capacidade anual — Solar vs Eólica")
        plt.xlabel("Ano")
        plt.ylabel("Fator de capacidade")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    # =====================================================
    # 3️⃣ LCOE E EROI
    # =====================================================
    if lcoe_solar_rpkwh and lcoe_eolica_rpkwh:
        plt.figure(figsize=(8, 5))
        plt.bar(["Solar", "Eólica"], [lcoe_solar_rpkwh, lcoe_eolica_rpkwh],
                color=[cores["solar"], cores["eolica"]])
        plt.title("LCOE — Custo nivelado (R$/kWh)")
        plt.ylabel("R$/kWh")
        plt.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    if eroi_solar and eroi_eolica:
        plt.figure(figsize=(8, 5))
        plt.bar(["Solar", "Eólica"], [eroi_solar, eroi_eolica],
                color=[cores["solar"], cores["eolica"]])
        plt.title("EROI — Retorno energético")
        plt.ylabel("EROI")
        plt.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

    # =====================================================
    # 4️⃣ PAYBACK
    # =====================================================
    if payback_solar or payback_eolica:
        plt.figure(figsize=(10, 6))
        plt.title("Payback acumulado — Solar vs Eólica")
        plt.xlabel("Ano")
        plt.ylabel("Fluxo acumulado (R$)")
        plt.grid(alpha=0.3)

        if payback_solar:
            anos_s = range(len(payback_solar["acumulado_nominal"]))
            plt.plot(anos_s, payback_solar["acumulado_nominal"],
                     label="Solar — Nominal", color=cores["solar"], linestyle="-", marker="o")
            plt.plot(anos_s, payback_solar["acumulado_descontado"],
                     label="Solar — Descontado", color=cores["solar"], linestyle="--")

        if payback_eolica:
            anos_e = range(len(payback_eolica["acumulado_nominal"]))
            plt.plot(anos_e, payback_eolica["acumulado_nominal"],
                     label="Eólica — Nominal", color=cores["eolica"], linestyle="-", marker="s")
            plt.plot(anos_e, payback_eolica["acumulado_descontado"],
                     label="Eólica — Descontado", color=cores["eolica"], linestyle="--")

        plt.legend()
        plt.tight_layout()
        plt.show()

    # =====================================================
    # 5️⃣ ECONOMIA FUTURA
    # =====================================================
    if payback_solar and producao_solar_anual:
        anos = np.arange(1, horizonte_anos + 1)
        energia_media_kwh = np.mean(list(producao_solar_anual.values()))
        capex_total = payback_solar["parametros"]["CAPEX_total"]
        om_anual = capex_total * payback_solar["parametros"]["OM_percentual_CAPEX"]

        prod_t = [energia_media_kwh * ((1 - 0.005) ** (t - 1)) for t in anos]
        receita = np.array(prod_t) * tarifa_r_kwh
        economia = np.cumsum(receita - om_anual) - capex_total

        cenarios = {"+1%/ano": 0.01, "+3%/ano": 0.03, "+5%/ano": 0.05}
        custos_rede = {}
        for lb, g in cenarios.items():
            tarifas = [tarifa_r_kwh * ((1 + g) ** (t - 1)) for t in anos]
            custos_rede[lb] = np.cumsum(np.array(tarifas) * energia_media_kwh)

        custo_const = np.cumsum([energia_media_kwh * tarifa_r_kwh for _ in anos])

        plt.figure(figsize=(10, 6))
        plt.title("Economia acumulada — Tarifa futura (20 anos)")
        plt.plot(anos, custo_const, color=cores["rede_const"], linestyle="--", label="Rede fixa")

        for lb, cor in zip(cenarios.keys(), [cores["rede_1"], cores["rede_3"], cores["rede_5"]]):
            plt.plot(anos, custos_rede[lb], color=cor, linewidth=2, label=f"Rede {lb}")

        plt.plot(anos, economia + custo_const[0],
                 color=cores["solar"], linewidth=2.5, label="Sistema solar")

        plt.xlabel("Ano")
        plt.ylabel("Valor acumulado (R$)")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.show()

    # =====================================================
    # 6️⃣ ROI REAL (NPV / CAPEX)
    # =====================================================
    def calc_roi(payback_data, cor, nome):
        CAPEX = payback_data["parametros"]["CAPEX_total"]

        NPV = payback_data["acumulado_descontado"][-1] + CAPEX
        roi_real = (NPV / CAPEX) * 100

        anos = np.arange(len(payback_data["acumulado_descontado"]))
        fluxo_desc = payback_data["acumulado_descontado"]
        roi_acum = (fluxo_desc / CAPEX) * 100
        roi_anual = np.diff(np.insert(roi_acum, 0, 0))
        roi_mov = pd.Series(roi_anual).rolling(window=3, min_periods=1).mean()

        plt.plot(anos, roi_acum, color=cor, linewidth=2.5, marker="o", label=f"{nome} — ROI acumulado (REAL)")
        plt.bar(anos, roi_anual, color=cor, alpha=0.25, label=f"{nome} — ROI anual (REAL)")
        plt.plot(anos, roi_mov, color=cor, linestyle="--", linewidth=1.8, label=f"{nome} — Média móvel (3a)")

        cruz = np.where(roi_acum >= 100)[0]
        if len(cruz) > 0:
            ano_pb = anos[cruz[0]]
            plt.scatter(ano_pb, 100, color=cor, s=80, edgecolor="black", zorder=5)
            plt.text(ano_pb + 0.3, 105, f"Payback ≈ {ano_pb} anos", color=cor, fontsize=10)

        print(f"💰 {nome.upper()} — ROI REAL = {roi_real:.2f}%  (NPV = {NPV:,.2f})")

        return roi_real, NPV

    if payback_solar or payback_eolica:
        plt.figure(figsize=(10, 6))
        plt.title("ROI REAL — Acumulado / Anual / Média móvel")
        plt.xlabel("Ano")
        plt.ylabel("ROI [%]")
        plt.grid(alpha=0.3)

        if payback_solar:
            roi_s, npv_s = calc_roi(payback_solar, cores["solar"], "Solar")
            roi_resultados["Solar"] = (roi_s, npv_s)

        if payback_eolica:
            roi_e, npv_e = calc_roi(payback_eolica, cores["eolica"], "Eólica")
            roi_resultados["Eólica"] = (roi_e, npv_e)

        plt.axhline(100, color="black", linestyle="--", linewidth=1.2, label="Payback (100%)")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # =====================================================
    # 7️⃣ ROI FINAL COMPARATIVO
    # =====================================================
    if roi_resultados:
        nomes = list(roi_resultados.keys())
        valores = [roi_resultados[n][0] for n in nomes]

        plt.figure(figsize=(8, 5))
        plt.bar(nomes, valores,
                color=[cores["solar"], cores["eolica"]][:len(nomes)],
                alpha=0.85)
        plt.axhline(100, color="black", linestyle="--", linewidth=1.2, label="Payback (100%)")
        plt.title("ROI REAL FINAL — Comparação Solar vs Eólica")
        plt.ylabel("ROI Total (%)")
        plt.grid(alpha=0.3, axis="y")
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
def score_inverse(a, b):
    """
    Inverse scoring for payback: smaller is better.
    Returns (score_a, score_b) in [0,1] summing to 1.
    Robust to inf, nan and zero-division.
    """
    # sanitize
    if a is None or b is None:
        return 0.5, 0.5
    a = float(a) if np.isfinite(a) or np.isinf(a) else np.nan
    b = float(b) if np.isfinite(b) or np.isinf(b) else np.nan

    # both nan
    if np.isnan(a) and np.isnan(b):
        return 0.5, 0.5

    # both inf => indistinguível
    if np.isinf(a) and np.isinf(b):
        return 0.5, 0.5

    # one inf wins (the finite one is better because payback smaller is better)
    if np.isinf(a) and not np.isinf(b):
        return 0.0, 1.0
    if np.isinf(b) and not np.isinf(a):
        return 1.0, 0.0

    # if one is nan, favor the finite one
    if np.isnan(a) and not np.isnan(b):
        return 0.0, 1.0
    if np.isnan(b) and not np.isnan(a):
        return 1.0, 0.0

    # now both finite numbers (>=0 expected). protect against zero
    a_safe = max(a, 1e-12)
    b_safe = max(b, 1e-12)
    inv_a = 1.0 / a_safe
    inv_b = 1.0 / b_safe
    s = inv_a + inv_b
    if s <= 0:
        return 0.5, 0.5
    return inv_a / s, inv_b / s


def definir_melhor_energia(
    solar_lcoe: float,
    eolica_lcoe: float,
    solar_eroi: float,
    eolica_eroi: float,
    solar_fc: float,
    eolica_fc: float,
    payback_solar: dict | None = None,
    payback_eolica: dict | None = None,
    pesos: dict | None = None,
    empate_tol: float = 1e-2,
    sigma_lcoe: dict | None = None,
    sigma_eroi: dict | None = None,
    sigma_fc: dict | None = None,
):
    """
    Scoring estável, transparente e sem divisões por zero:
     - LCOE: score inverso relativo (menor melhor) com saturação
     - EROI/FC: score relativo por norma ao máximo entre ambos
     - Payback: similar ao LCOE (menor melhor)
     - Penalidades por sigma aplicadas multiplicativamente (reduz confiança)
    """
    # defensiva
    def _safe_num(x):
        try:
            return float(x)
        except Exception:
            return np.nan

    solar_lcoe = _safe_num(solar_lcoe)
    eolica_lcoe = _safe_num(eolica_lcoe)
    solar_eroi = _safe_num(solar_eroi)
    eolica_eroi = _safe_num(eolica_eroi)
    solar_fc = _safe_num(solar_fc)
    eolica_fc = _safe_num(eolica_fc)

    # pesos normalizados
    if pesos is None:
        pesos = {"lcoe": 0.4, "eroi": 0.3, "fc": 0.2, "payback": 0.1}
    total_p = sum(pesos.values()) or 1.0
    pesos = {k: v / total_p for k, v in pesos.items()}

    sigma_lcoe = sigma_lcoe or {"solar": 0.0, "eolica": 0.0}
    sigma_eroi = sigma_eroi or {"solar": 0.0, "eolica": 0.0}
    sigma_fc = sigma_fc or {"solar": 0.0, "eolica": 0.0}

    # --- LCOE score: mapping to [0,1], lower better ---
    # use robust ratio with epsilon
    eps = 1e-12
    max_lcoe = np.nanmax([solar_lcoe if not np.isnan(solar_lcoe) else 0.0,
                          eolica_lcoe if not np.isnan(eolica_lcoe) else 0.0, eps])
    # invert and normalize
    inv_s = 1.0 / (solar_lcoe + eps) if not np.isnan(solar_lcoe) else 0.0
    inv_e = 1.0 / (eolica_lcoe + eps) if not np.isnan(eolica_lcoe) else 0.0
    s_sum = inv_s + inv_e
    if s_sum <= 0:
        score_lcoe_solar = score_lcoe_eolica = 0.5
    else:
        score_lcoe_solar = inv_s / s_sum
        score_lcoe_eolica = inv_e / s_sum

    # --- EROI and FC: relative to max (higher better) ---
    def _rel_score(a, b):
        a = 0.0 if (a is None or (isinstance(a, float) and np.isnan(a))) else float(a)
        b = 0.0 if (b is None or (isinstance(b, float) and np.isnan(b))) else float(b)
        s = a + b
        if s <= 0:
            return 0.5, 0.5
        return a / s, b / s

    score_eroi_solar, score_eroi_eolica = _rel_score(solar_eroi, eolica_eroi)
    score_fc_solar, score_fc_eolica = _rel_score(solar_fc, eolica_fc)

    # --- Payback (descontado) score: menor melhor. None -> neutral 0.5
    pb_s = payback_solar.get("payback_descontado_anos") if payback_solar else None
    pb_e = payback_eolica.get("payback_descontado_anos") if payback_eolica else None

    def _payback_score(a, b):
        if a is None and b is None:
            return 0.5, 0.5
        if a is None:
            return 0.25, 0.75
        if b is None:
            return 0.75, 0.25
        a_f = float(a)
        b_f = float(b)
        # invert same as LCOE approach
        inv_a = 1.0 / (a_f + eps)
        inv_b = 1.0 / (b_f + eps)
        s = inv_a + inv_b
        if s <= 0:
            return 0.5, 0.5
        return inv_a / s, inv_b / s

    score_pb_solar, score_pb_eolica = _payback_score(pb_s, pb_e)

    # apply risk penalization: multiplicative decay exp(-k * sigma)
    def penalize(s, sigma, k=2.0):
        sigma = float(sigma or 0.0)
        return s * np.exp(-k * sigma)

    score_lcoe_solar = penalize(score_lcoe_solar, sigma_lcoe.get("solar", 0.0))
    score_lcoe_eolica = penalize(score_lcoe_eolica, sigma_lcoe.get("eolica", 0.0))
    score_eroi_solar = penalize(score_eroi_solar, sigma_eroi.get("solar", 0.0))
    score_eroi_eolica = penalize(score_eroi_eolica, sigma_eroi.get("eolica", 0.0))
    score_fc_solar = penalize(score_fc_solar, sigma_fc.get("solar", 0.0))
    score_fc_eolica = penalize(score_fc_eolica, sigma_fc.get("eolica", 0.0))

    # normalize each pair again to avoid drift
    def _renorm_pair(a, b):
        s = a + b
        if s <= 0:
            return 0.5, 0.5
        return a / s, b / s

    score_lcoe_solar, score_lcoe_eolica = _renorm_pair(score_lcoe_solar, score_lcoe_eolica)
    score_eroi_solar, score_eroi_eolica = _renorm_pair(score_eroi_solar, score_eroi_eolica)
    score_fc_solar, score_fc_eolica = _renorm_pair(score_fc_solar, score_fc_eolica)
    score_pb_solar, score_pb_eolica = _renorm_pair(score_pb_solar, score_pb_eolica)

    # weighted total
    total_solar = (
        score_lcoe_solar * pesos["lcoe"]
        + score_eroi_solar * pesos["eroi"]
        + score_fc_solar * pesos["fc"]
        + score_pb_solar * pesos["payback"]
    )
    total_eolica = (
        score_lcoe_eolica * pesos["lcoe"]
        + score_eroi_eolica * pesos["eroi"]
        + score_fc_eolica * pesos["fc"]
        + score_pb_eolica * pesos["payback"]
    )

    diff = total_solar - total_eolica
    margem = abs(diff)
    if margem <= empate_tol:
        recomendacao = "empate/indefinido"
        confidence = 0.0
    else:
        recomendacao = "solar" if diff > 0 else "eolica"
        confidence = float(np.clip(abs(diff) / (abs(total_solar) + abs(total_eolica) + 1e-12), 0.0, 1.0) * 100.0)

    detalhes = {
        "lcoe": "Solar" if solar_lcoe < eolica_lcoe else "Eólica" if eolica_lcoe < solar_lcoe else "igual",
        "eroi": "Solar" if solar_eroi > eolica_eroi else "Eólica" if eolica_eroi > solar_eroi else "igual",
        "fc": "Solar" if solar_fc > eolica_fc else "Eólica" if eolica_fc > solar_fc else "igual",
        "payback": (
            "Solar" if (pb_s is not None and (pb_e is None or pb_s < pb_e))
            else "Eólica" if (pb_e is not None and (pb_s is None or pb_e < pb_s))
            else "igual"
        ),
    }

    alertas = []
    if solar_fc > 1.2 or eolica_fc > 1.2:
        alertas.append("Fator de capacidade acima do limite físico (>1.2).")

    return {
        "scores": {
            "solar": {
                "lcoe": score_lcoe_solar,
                "eroi": score_eroi_solar,
                "fc": score_fc_solar,
                "payback": score_pb_solar,
                "total": total_solar,
            },
            "eolica": {
                "lcoe": score_lcoe_eolica,
                "eroi": score_eroi_eolica,
                "fc": score_fc_eolica,
                "payback": score_pb_eolica,
                "total": total_eolica,
            },
        },
        "recomendacao": recomendacao,
        "margem_numerica": margem,
        "confidence": confidence,
        "detalhes": detalhes,
        "sigma": {"lcoe": sigma_lcoe, "eroi": sigma_eroi, "fc": sigma_fc},
        "alertas": alertas,
    }


In [ ]:
# ================================================================
# SANITIZAÇÃO FINAL (TZ/ordenamento) — assinatura preservada
# ================================================================
def _sanitize_date(df: pd.DataFrame, timezone: str) -> pd.DataFrame:
    """
    Normaliza e blinda o dataframe contra ambiguidades em 'date' e índice temporal.
    Preserva interface.
    """
    out = ensure_no_date_ambiguity(df.copy())

    # Se índice for DatetimeIndex, trazer para coluna 'date'
    if isinstance(out.index, pd.DatetimeIndex):
        try:
            idx = out.index.tz_convert(timezone)
        except Exception:
            idx = out.index
        out["date"] = pd.to_datetime(idx)
        out.reset_index(drop=True, inplace=True)

    # Normaliza 'date'
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"], errors="coerce")
        out = ensure_no_date_ambiguity(out)

    out["date"] = out["date"].dt.tz_localize(None).dt.floor("h")
    out.sort_values("date", inplace=True)
    out["ano"] = out["date"].dt.year
    return ensure_no_date_ambiguity(out)


import logging
import sys
import traceback
from datetime import datetime

# ---------------------------
# Helper: add TRACE level
# ---------------------------
TRACE_LEVEL_NUM = 5
logging.addLevelName(TRACE_LEVEL_NUM, "TRACE")


def trace(self, message, *args, **kws):
    if self.isEnabledFor(TRACE_LEVEL_NUM):
        # Yes, logger takes its '*args' as 'args'.
        self._log(TRACE_LEVEL_NUM, message, args, **kws)


logging.Logger.trace = trace  # type: ignore

# ---------------------------
# Configure logger (TRACE unlimited)
# ---------------------------
logger = logging.getLogger("energy_analysis")
logger.setLevel(TRACE_LEVEL_NUM)
ch = logging.StreamHandler(sys.stdout)
ch.setLevel(TRACE_LEVEL_NUM)
formatter = logging.Formatter("%(asctime)s | %(levelname)-5s | %(name)s | %(message)s", datefmt="%H:%M:%S")
ch.setFormatter(formatter)
logger.handlers = []
logger.addHandler(ch)

# Optional: also log warnings/errors to stderr
eh = logging.StreamHandler(sys.stderr)
eh.setLevel(logging.WARNING)
eh.setFormatter(formatter)
logger.addHandler(eh)

# ---------------------------
# MAIN instrumentado (TRACE ilimitado)
# ---------------------------
def _compute_capex_total(equipamento_cost, instalacao_cost, quantidade, potencia_kw, role="generic", turbina_meta=None):
    """
    Heurística segura para CAPEX_total (revisada):
      - Prioriza overrides explícitos: 'capex_override' (total) ou 'capex_per_kw' (R$/kW).
      - Detecta custo unitário plausível (por unidade) e custo por kW plausível.
      - Evita multiplicadores arbitrários e entrega R$/projeto coerente.
    """
    # override total do projeto
    if turbina_meta and "capex_override" in turbina_meta:
        return float(turbina_meta["capex_override"])

    # override por potência (R$/kW)
    if turbina_meta and "capex_per_kw" in turbina_meta:
        capex_per_kw = float(turbina_meta["capex_per_kw"])
        potencia_kw = float(potencia_kw or 0.0)
        return float(capex_per_kw * max(1.0, potencia_kw))

    # normalizar inputs
    equipamento_cost = float(equipamento_cost or 0.0)
    instalacao_cost = float(instalacao_cost or 0.0)
    quantidade = int(max(1, quantidade or 1))
    potencia_kw = float(potencia_kw or 0.0)

    # heurísticas:
    # - Se custo unitário muito alto (>1e6) tratamos como valor já total por equipamento (multiplicar por quantidade).
    # - Se custo unitário moderado (<1e6) tratamos como custo por unidade e multiplicamos por quantidade.
    # - Se potência instalada (kW) é informada e equipamento_cost parece pequeno (p.ex. <1e5) podemos também estimar via R$/kW fallback.
    if equipamento_cost >= 1e6:
        # custo unitário grande (provavelmente por turbina utility-scale em escala): multiplicar por quantidade
        base_unit_cost = equipamento_cost + instalacao_cost
        total_base = base_unit_cost * quantidade
    else:
        # custo típico por unidade (painel/turbina pequena)
        base_unit_cost = equipamento_cost + instalacao_cost
        total_base = base_unit_cost * quantidade

    # fallback: se potência informada e total_base aparenta muito baixo por kW, tentar estimativa por R$/kW sensata
    if potencia_kw > 0:
        implied_r_per_kw = total_base / max(potencia_kw, 1e-6)
        # se implied < 100 (R$/kW) ou > 100000 (R$/kW) é improvável -> usar faixas referência por tecnologia
        if implied_r_per_kw < 100.0 or implied_r_per_kw > 100000.0:
            # defaults plausíveis (podem ser sobrescritos via turbina_meta)
            defaults = {
                "solar": 3500.0,     # R$/kW (utility-scale, exemplo)
                "eolica": 9000.0,
                "generic": 4000.0
            }
            tipo = (turbina_meta.get("type") if turbina_meta else role) if turbina_meta else role
            capex_per_kw_default = float(defaults.get(str(tipo).lower(), defaults["generic"]))
            total_base = capex_per_kw_default * max(1.0, potencia_kw)

    # sanity clamp: não permitir valores negativos ou zero
    capex_total = float(max(total_base, 0.0))

    return capex_total


def main_trace():
    try:
        logger.trace("=== Início da execução principal (TRACE ilimitado) ===")

        # ---------- CONFIGURAÇÃO BÁSICA ----------
        # localização e período
        # ---------- LOCALIZAÇÃO ----------
        # EOLICA
        #latitude, longitude = -29.9200000, -50.2700000
        #latitude, longitude = -5.2000000, -37.0200000
        #latitude, longitude = -3.2700000, -39.2700000
        #latitude, longitude = -8.5200000, -41.5300000
        #latitude, longitude = -3.3500000, -39.8000000

        # SOLAR
        #latitude, longitude = -14.0700000, -42.4900000 !!!
        #latitude, longitude = -9.400, -38.300   # Paulo Afonso - BA (~5.8 kWh/m²·dia) >>>
        latitude, longitude = -10.400, -39.300  # Monte Santo - BA (~6.0 kWh/m²·dia)
        #latitude, longitude = -12.600, -41.400  # Lençóis - BA (~5.9 kWh/m²·dia)
        #latitude, longitude = -7.300, -39.300   # Cariri - CE (~5.7 kWh/m²·dia) $$
        #latitude, longitude = -5.200, -40.700   # Crateús - CE (~5.6 kWh/m²·dia)
        #latitude, longitude = -6.400, -36.800   # Cruzeta - RN (~5.5 kWh/m²·dia) !!!
        #latitude, longitude = -14.100, -42.500  # Caetité - BA (~5.8 kWh/m²·dia)
        #latitude, longitude = -3.700, -40.400   # Sobral - CE (~5.4 kWh/m²·dia) $$
        #latitude, longitude = -2.500, -44.300   # São Luís - MA (~5.4 kWh/m²·dia)
        #latitude, longitude = -12.9618813, -38.5000007  # Salvador - BA (~5.3 kWh/m²·dia)
        start_date, end_date = "20100101", "20250101"
        timezone = "America/Sao_Paulo"

        logger.trace(f"[CONFIG] latitude={latitude}, longitude={longitude}, start={start_date}, end={end_date}, tz={timezone}")

        # seleção equipamentos (índices)
        indice_painel_escolhido = 1   # 0=AmeriSolar, 1=WEG HMM1, 2=WEG HMM0
        indice_turbina_escolhida = 0  # 0=Bornay 13+, 1=WEG AGW-172-7MW

        # Bottle-neck: ensure PAINEL/TURBINA are defined in global scope
        try:
            PAINEL = PAINEIS_SOLARES[indice_painel_escolhido]
            TURBINA = TURBINAS_EOLICAS[indice_turbina_escolhida]

            # plausibility check: relativas de potência instalada
            potencia_total_solar_kw = float(PAINEL.get("potencia_kwp", 0.0)) * int(PAINEL.get("quantidade_paineis", 1))
            potencia_total_eolica_kw = float(TURBINA.get("potencia_nominal", 0.0)) * int(TURBINA.get("quantidade_turbinas", 1))
            if potencia_total_eolica_kw == 0:
                ratio = float("inf")
            else:
                ratio = potencia_total_solar_kw / potencia_total_eolica_kw
            if ratio > 10.0:
                logger.warning("[PLAUSIBILITY] Desbalanceamento de escala detectado: solar/eolica pot ratio = %.2f (recomendo comparar por kW instalado ou ajustar quantidades).", ratio)

        except Exception as e:
            logger.exception("Falha ao selecionar equipamentos: %s", e)
            raise

        logger.trace("PAINEL escolhido: %s", PAINEL)
        logger.trace("TURBINA escolhida: %s", TURBINA)

        # ---------- COLETA DE DADOS CLIMÁTICOS ----------
        logger.trace("[GET] Chamando search_info_weather()")
        t0 = datetime.utcnow()

        hourly_data = search_info_weather(
            latitude,
            longitude,
            start_date,
            end_date,
            timezone,
            cache_path=".cache",
            cache_expire_hours=24,
            normalize=True,
            use_multisource=True
        )

        print(f"✅ Multi-fonte: {len(hourly_data)} registros")
        print(f"dt_hours: {hourly_data['dt_hours']}")

        # Mostrar primeiras linhas
        print("\nPrimeiras 5 linhas:")
        print(hourly_data['data'].head())

        t1 = datetime.utcnow()
        logger.trace("[GET] search_info_weather retornou em %s segundos", (t1 - t0).total_seconds())
        logger.trace("[GET] keys retorno: %s", list(hourly_data.keys()))

        # quick sanity of hourly_data
        if not isinstance(hourly_data, dict) or "data" not in hourly_data:
            logger.error("Formato inesperado de hourly_data")
            raise ValueError("Formato inesperado de hourly_data")

        # ---------------- SANITIZAÇÃO / TIMEZONE ----------------
        logger.trace("[SANITIZE] Chamando _sanitize_date() com DataFrame")
        df_raw = pd.DataFrame(hourly_data["data"])
        logger.trace("[SANITIZE] df_raw.shape=%s", df_raw.shape)
        df = _sanitize_date(df_raw, timezone)

        print(df[['wind_speed_10m','wind_speed_100m']].head(20))

        logger.trace("[SANITIZE] df_sanitizado.shape=%s | date_min=%s | date_max=%s",
                     df.shape, df['date'].min() if 'date' in df.columns else None, df['date'].max() if 'date' in df.columns else None)

        # ---------- VARIABILIDADE (sigma) ----------
        logger.trace("[VAR] Calculando variabilidade inter-anuais")
        rad = df.groupby("ano")["shortwave_radiation"].mean()
        wind = df.groupby("ano")["wind_speed_10m"].mean()
        cv_rad = float(rad.std() / rad.mean()) if rad.mean() else 0.0
        cv_wind = float(wind.std() / wind.mean()) if wind.mean() else 0.0
        n_anos = len(rad)
        se_rad = cv_rad / (n_anos ** 0.5) if n_anos > 0 else 0.0
        se_wind = cv_wind / (n_anos ** 0.5) if n_anos > 0 else 0.0
        sigma_lcoe = {"solar": se_rad * 0.5, "eolica": se_wind * 0.5}
        sigma_eroi = {"solar": se_rad, "eolica": se_wind}
        sigma_fc = {"solar": se_rad, "eolica": se_wind}

        logger.trace("[VAR] cv_rad=%s cv_wind=%s n_anos=%s | sigma_lcoe=%s",
                     cv_rad, cv_wind, n_anos, sigma_lcoe)

        # ---------- PRODUÇÃO ANUAL E FC ----------
        logger.trace("[PROD] Chamando calcular_producao_e_fator_capacidade_anual() - SOLAR")
        solar_prod_dict, solar_fc_dict, solar_media_prod, solar_media_fc, solar_detalhes = \
            calcular_producao_e_fator_capacidade_anual(df, "solar", painel=PAINEL, devolver_detalhes=True)
        logger.trace("[PROD][SOLAR] anos=%s | producoes=%s", list(solar_prod_dict.keys()), {k: solar_prod_dict[k] for k in list(solar_prod_dict.keys())[:5]})
        logger.trace("[PROD][SOLAR] media_prod=%s fc_media=%s", solar_media_prod, solar_media_fc)

        logger.trace("[PROD] Chamando calcular_producao_e_fator_capacidade_anual() - EOLICA")
        # NOTE: calcular_producao_e_fator_capacidade_anual deve agora usar tanto wind_speed_10m quanto wind_speed_100m internamente.
        eolica_prod_dict, eolica_fc_dict, eolica_media_prod, eolica_media_fc, eolica_detalhes = \
            calcular_producao_e_fator_capacidade_anual(df, "eolica", turbina=TURBINA, devolver_detalhes=True)
        logger.trace("[PROD][EOLICA] anos=%s | producoes=%s", list(eolica_prod_dict.keys()), {k: eolica_prod_dict[k] for k in list(eolica_prod_dict.keys())[:5]})
        logger.trace("[PROD][EOLICA] media_prod=%s fc_media=%s", eolica_media_prod, eolica_media_fc)

        print("\n[DEBUG MAIN] PRODUÇÃO ANUAL SOLAR:", solar_media_prod)
        print("[DEBUG MAIN] PRODUÇÃO ANUAL EOLICA:", eolica_media_prod)

        # sanity: check lengths and sign
        logger.trace("[SANITY] solar_prod_len=%d eolica_prod_len=%d", len(solar_prod_dict), len(eolica_prod_dict))
        if solar_media_prod <= 0:
            logger.error("Produção média solar inválida (<=0)")
        if eolica_media_prod <= 0:
            logger.warning("Produção média eólica muito baixa (<=0) — revisar dados/turbina")

        # ---------- CAPEX (total) ----------
        # Agora o CAPEX é calculado por heurística realista e parametrizável.
        CAPEX_SOLAR = _compute_capex_total(
            equipamento_cost=PAINEL.get("custo_equipamento", 0.0),
            instalacao_cost=PAINEL.get("custo_instalacao", 0.0),
            quantidade=PAINEL.get("quantidade_paineis", 1),
            potencia_kw=PAINEL.get("potencia_kwp", 0.0) * PAINEL.get("quantidade_paineis", 1),
            role="solar",
            turbina_meta=PAINEL
        )
        CAPEX_EOLICA = _compute_capex_total(
            equipamento_cost=TURBINA.get("custo_equipamento", 0.0),
            instalacao_cost=TURBINA.get("custo_instalacao", 0.0),
            quantidade=TURBINA.get("quantidade_turbinas", 1),
            potencia_kw=TURBINA.get("potencia_nominal", 0.0) * TURBINA.get("quantidade_turbinas", 1),
            role="eolica",
            turbina_meta=TURBINA
        )
        logger.trace("[CAPEX] CAPEX_SOLAR=%s | CAPEX_EOLICA=%s", CAPEX_SOLAR, CAPEX_EOLICA)

        # ---------- LCOE econômico (coerente com payback) ----------
        tarifa_r_kwh = 0.70
        logger.trace("[LCOE] Chamando calcular_lcoe() - SOLAR")
        lcoe_solar = calcular_lcoe(
            producoes_anuais=solar_prod_dict,
            capex_total=CAPEX_SOLAR,
            vida_util=int(PAINEL.get("vida_util", 25)),
            tarifa_r_kwh=tarifa_r_kwh,
            degradacao_anual=float(PAINEL.get("degradacao_anual", 0.005)),
            om_percentual_capex=float(PAINEL.get("om_percentual_capex", 0.01)),
            taxa_desconto=0.08,
            logger=logger
        )
        logger.trace("[LCOE][SOLAR] retorno=%s", lcoe_solar)

        logger.trace("[LCOE] Chamando calcular_lcoe() - EOLICA")
        lcoe_eolica = calcular_lcoe(
            producoes_anuais=eolica_prod_dict,
            capex_total=CAPEX_EOLICA,
            vida_util=int(TURBINA.get("vida_util", 25)),
            tarifa_r_kwh=tarifa_r_kwh,
            degradacao_anual=float(TURBINA.get("degradacao_anual", 0.005)),
            om_percentual_capex=float(TURBINA.get("om_percentual_capex", 0.03)),
            taxa_desconto=0.08,
            logger=logger
        )
        logger.trace("[LCOE][EOLICA] retorno=%s", lcoe_eolica)

        # ---------- EROI ----------
        logger.trace("[EROI] Chamando calcular_eroi() - SOLAR")
        eroi_solar = calcular_eroi(
            potencia_kw=PAINEL["potencia_kwp"] * PAINEL["quantidade_paineis"],
            vida_util=int(PAINEL.get("vida_util", 25)),
            producao_anual_kwh=solar_media_prod,
            degradacao_anual=float(PAINEL.get("degradacao_anual", 0.005)),
            fonte="solar",
            return_details=True
        )
        logger.trace("[EROI][SOLAR] %s", eroi_solar)

        logger.trace("[EROI] Chamando calcular_eroi() - EOLICA")
        eroi_eolica = calcular_eroi(
            potencia_kw=TURBINA["potencia_nominal"] * TURBINA["quantidade_turbinas"],
            vida_util=int(TURBINA.get("vida_util", 25)),
            producao_anual_kwh=eolica_media_prod,
            degradacao_anual=float(TURBINA.get("degradacao_anual", 0.005)),
            fonte="eolica",
            return_details=True
        )
        logger.trace("[EROI][EOLICA] %s", eroi_eolica)

        # ---------- PAYBACK (nominal + descontado) ----------
        logger.trace("[PAYBACK] Chamando calcular_payback() - SOLAR")
        payback_solar = calcular_payback(
            capex_total=CAPEX_SOLAR,
            producao_anual_kwh=solar_media_prod,
            tarifa_r_kwh=tarifa_r_kwh,
            vida_util=int(PAINEL.get("vida_util", 25)),
            degradacao_anual=float(PAINEL.get("degradacao_anual", 0.005)),
            om_fixo_anual=float(PAINEL.get("om_fixo_anual", 0.0)),
            om_percentual=float(PAINEL.get("om_percentual_capex", 0.01)),
            taxa_desconto=0.08,
            return_cashflow=True
        )
        logger.trace("[PAYBACK][SOLAR] %s", {k: (payback_solar[k] if k in payback_solar else None) for k in ["payback_nominal_anos", "payback_descontado_anos"]})

        logger.trace("[PAYBACK] Chamando calcular_payback() - EOLICA")
        payback_eolica = calcular_payback(
            capex_total=CAPEX_EOLICA,
            producao_anual_kwh=eolica_media_prod,
            tarifa_r_kwh=tarifa_r_kwh,
            vida_util=int(TURBINA.get("vida_util", 25)),
            degradacao_anual=float(TURBINA.get("degradacao_anual", 0.005)),
            om_fixo_anual=float(TURBINA.get("om_fixo_anual", 0.0)),
            om_percentual=float(TURBINA.get("om_percentual_capex", 0.03)),
            taxa_desconto=0.08,
            return_cashflow=True
        )
        logger.trace("[PAYBACK][EOLICA] %s", {k: (payback_eolica[k] if k in payback_eolica else None) for k in ["payback_nominal_anos", "payback_descontado_anos"]})

        # ---------- DECISÃO final (definir_melhor_energia) ----------
        logger.trace("[DECISAO] Chamando definir_melhor_energia()")
        decisao = definir_melhor_energia(
            solar_lcoe=lcoe_solar["LCOE"],
            eolica_lcoe=lcoe_eolica["LCOE"],
            solar_eroi=eroi_solar["EROI"] if isinstance(eroi_solar, dict) else eroi_solar,
            eolica_eroi=eroi_eolica["EROI"] if isinstance(eroi_eolica, dict) else eroi_eolica,
            solar_fc=solar_media_fc,
            eolica_fc=eolica_media_fc,
            payback_solar=payback_solar,
            payback_eolica=payback_eolica,
            sigma_lcoe=sigma_lcoe,
            sigma_eroi=sigma_eroi,
            sigma_fc=sigma_fc
        )
        logger.trace("[DECISAO] %s", decisao)

        # ---------- LOG FINAL resumido (mas com TRACE já disponível) ----------
        logger.info("=== RESULTADOS RESUMIDOS ===")
        logger.info("Produção média anual (solar): %0.2f kWh | (eólica): %0.2f kWh", solar_media_prod, eolica_media_prod)
        logger.info("Fator de capacidade (solar): %0.3f | (eólica): %0.3f", solar_media_fc, eolica_media_fc)
        logger.info("LCOE econômico (solar): %0.4f R$/kWh | (eólica): %0.4f R$/kWh", lcoe_solar["LCOE"], lcoe_eolica["LCOE"])
        logger.info("NPV (solar): %0.2f | (eólica): %0.2f", lcoe_solar["NPV"], lcoe_eolica["NPV"])
        logger.info("Payback descontado (solar): %0.2f anos | (eólica): %s anos",
                    payback_solar["payback_descontado_anos"],
                    (payback_eolica["payback_descontado_anos"] if np.isfinite(payback_eolica["payback_descontado_anos"]) else "∞"))
        logger.info("Decisão final: %s (confiança: %0.2f%%) — detalhes: %s",
                    decisao["recomendacao"].upper(), decisao["confidence"], decisao["detalhes"])

        # ---------- GRAFICOS (opcional) ----------
        try:
            logger.trace("[PLOT] Chamando gerar_graficos()")
            gerar_graficos(
                producao_solar_anual=solar_prod_dict,
                producao_eolica_anual=eolica_prod_dict,
                fatores_capacidade_solar=solar_fc_dict,
                fatores_capacidade_eolica=eolica_fc_dict,
                lcoe_solar_rpkwh=lcoe_solar["LCOE"],
                lcoe_eolica_rpkwh=lcoe_eolica["LCOE"],
                eroi_solar=eroi_solar["EROI"] if isinstance(eroi_solar, dict) else eroi_solar,
                eroi_eolica=eroi_eolica["EROI"] if isinstance(eroi_eolica, dict) else eroi_eolica,
                lcoe_solar_details=lcoe_solar,
                lcoe_eolica_details=lcoe_eolica,
                investimentos={"solar": CAPEX_SOLAR, "eolica": CAPEX_EOLICA},
                payback_solar=payback_solar,
                payback_eolica=payback_eolica,
                tarifa_r_kwh=tarifa_r_kwh,
                horizonte_anos=20,
            )
            logger.trace("[PLOT] gerar_graficos finalizado")
        except Exception:
            logger.exception("Falha ao gerar gráficos (não fatal)")

        logger.trace("=== Execução principal concluída ===")

        return {
            "solar": {
                "producoes_anuais": solar_prod_dict,
                "media_producao": solar_media_prod,
                "fc_media": solar_media_fc,
                "lcoe": lcoe_solar,
                "eroi": eroi_solar,
                "payback": payback_solar
            },
            "eolica": {
                "producoes_anuais": eolica_prod_dict,
                "media_producao": eolica_media_prod,
                "fc_media": eolica_media_fc,
                "lcoe": lcoe_eolica,
                "eroi": eroi_eolica,
                "payback": payback_eolica
            },
            "decisao": decisao,
            "sigma": {"lcoe": sigma_lcoe, "eroi": sigma_eroi, "fc": sigma_fc}
        }

    except Exception as exc:
        logger.exception("FALHA FATAL no main: %s", exc)
        traceback.print_exc()
        raise

# ---------------------------
# Execute main_trace quando chamado diretamente
# ---------------------------
if __name__ == "__main__":
    resultado = main_trace()
    # se quiser persistir resultado em arquivo:
    try:
        import json
        with open("energy_analysis_result.json", "w", encoding="utf-8") as f:
            json.dump(resultado, f, default=str, indent=2)
        logger.trace("[IO] Resultado salvo em energy_analysis_result.json")
    except Exception:
        logger.exception("Falha ao salvar resultado (não fatal)")


## 📌 McER (version 2)

In [83]:
!pip install pandas numpy
!pip install pvlib
!pip install windpowerlib
!pip install pyarrow
!pip install numpy-financial

### 1. Configuração e Definições de Tipos (config.py)


In [84]:
"""
Configuração central do sistema com parâmetros atualizados para 2024.
"""
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from datetime import datetime
from enum import Enum

class TipoSistema(Enum):
    SOLAR = "solar"
    EOLICO = "eolico"
    HIBRIDO = "hibrido"

@dataclass
class Localizacao:
    """Localização geográfica com validação."""
    latitude: float  # -90 a 90 graus
    longitude: float  # -180 a 180 graus
    altitude: float = 0.0  # metros
    timezone: str = "America/Sao_Paulo"
    nome: str = ""
    codigo: str = ""

    def __post_init__(self):
        if not -90 <= self.latitude <= 90:
            raise ValueError(f"Latitude inválida: {self.latitude}")
        if not -180 <= self.longitude <= 180:
            raise ValueError(f"Longitude inválida: {self.longitude}")

from dataclasses import dataclass, field
from typing import Dict, List
import numpy as np

@dataclass
class PainelSolar:
    """Parâmetros de módulo fotovoltaico atualizados 2024 com curva de potência simplificada."""
    fabricante: str = "Jinko Solar"
    modelo: str = "Tiger Neo 550W"
    potencia_nominal_w: float = 550.0  # W STC
    eficiencia_stc: float = 0.215
    area_m2: float = 2.58
    coef_temperatura_pmax: float = -0.0034
    vida_util: int = 30
    degradacao_anual: float = 0.004

    # Parâmetros elétricos STC (datasheet)
    v_mp_stc: float = 40.9
    i_mp_stc: float = 13.45
    v_oc_stc: float = 49.62
    i_sc_stc: float = 14.03

    # Coeficientes de temperatura (aproximados)
    aisc: float = 0.0005   # A/°C
    aimp: float = 0.0005   # A/°C
    b_voco: float = -0.28  # V/°C
    b_vmpo: float = -0.35  # V/°C

    # Diode factor and cell count estimados
    n: float = 1.3
    cells_in_series: int = 144

    @property
    def parametros_pvlib(self) -> Dict:
        # Usados diretamente pelo pvlib
        return {
            'pdc0': self.potencia_nominal_w,
            'gamma_pdc': self.coef_temperatura_pmax,
            'area': self.area_m2,
            'technology': 'crystSi',
            'C0': 0.0,                 # estimativa inicial
            'C1': 0.0,
            'C2': 0.0,
            'C3': 0.0,
            'Isco': self.i_sc_stc,
            'Impo': self.i_mp_stc,
            'Voco': self.v_oc_stc,
            'Vmpo': self.v_mp_stc,
            'Aisc': self.aisc,
            'Aimp': self.aimp,
            'Bvoco': self.b_voco,
            'Mbvoc': 0.0,
            'Bvmpo': self.b_vmpo,
            'Mbvmp': 0.0,
            'N': self.n,
            'Cells_in_Series': self.cells_in_series,
        }

@dataclass
class TurbinaEolica:
    """Parâmetros de turbina eólica atualizados 2024."""
    fabricante: str = "Vestas"
    modelo: str = "V52-850kW"
    potencia_nominal_kw: float = 850.0  # kW
    diametro_rotor_m: float = 52.0  # m
    altura_cubo_m: float = 50.0  # m
    velocidade_cut_in: float = 4.0  # m/s
    velocidade_nominal: float = 14.0  # m/s
    velocidade_cut_out: float = 25.0  # m/s
    vida_util: int = 20  # anos
    degradacao_anual: float = 0.008  # %/ano

    @property
    def area_varredura(self) -> float:
        """Área de varredura do rotor (m²)."""
        return 3.14159 * (self.diametro_rotor_m / 2) ** 2

    @property
    def curva_potencia(self) -> Dict[str, list]:
        """
        Curva de potência discretizada para Vestas V52-850 kW (valores indicativos).
        Velocidades em m/s, potências em W.
        Fonte: especificações públicas e curvas típicas de desempenho. :contentReference[oaicite:3]{index=3}
        """
        # Tabela aproximada (wind_speed → power_W)
        # A forma segue uma rampa típica: cresce com vento entre cut-in e nominal,
        # depois mantém ~potência nominal, e cai a zero após cut-out.
        return {
            "wind_speed": [
                0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
                10, 11, 12, 13, 14, 15, 16, 17,
                18, 19, 20, 21, 22, 23, 24, 25
            ],
            "value": [
                0, 0, 0, 0, 25500, 67400, 125000, 203000,
                304000, 425000, 554000, 671000, 759000, 811000,
                836000, 846000, 849000, 850000,
                850000, 850000, 850000, 850000, 850000, 850000,
                850000, 850000
            ]
        }

@dataclass
class ConfiguracaoSistema:
    """Configuração geral do sistema com parâmetros 2024."""
    nome_projeto: str = "Análise Comparativa 15 Locais"
    horizonte_temporal: int = 25  # anos
    taxa_desconto_real: float = 0.06  # 6% a.a.
    taxa_cambio_usd_brl: float = 5.0  # R$/USD

    # Critérios de decisão
    criterios_decisao: Dict = field(default_factory=lambda: {
        'limiar_vento_eolico': 4.0,     # m/s para considerar eólica
        'limiar_vento_excelente': 5.0,   # m/s para eólica excelente
        'limiar_fc_eolico': 0.25,       # 25% fator capacidade mínimo
        'limiar_ghi_solar': 180.0,      # W/m² para considerar solar
        'diferenca_minima_hibrido': 0.15,  # Diferença de pontuação para híbrido
        'pesos': {
            'lcoe': 0.15,           # Reduzido
            'npv': 0.10,            # Reduzido
            'tir': 0.08,            # Reduzido
            'fator_capacidade': 0.25, # Aumentado
            'producao_anual': 0.20,   # Aumentado
            'investimento': 0.05,    # Reduzido
            'payback': 0.07,         # Reduzido
            'velocidade_vento': 0.10 # Aumentado
        }
    })

    # Parâmetros econômicos atualizados (IRENA 2024, EPE Brasil)
    parametros_economicos: Dict = field(default_factory=lambda: {
        'solar': {
            'capex_usd_kw': 550.0,       # USD/kW (R$ 2.750/kW)
            'capex_min_usd_kw': 550.0,   # USD/kW mínimo
            'capex_max_usd_kw': 850.0,   # USD/kW máximo
            'opex_percent_capex': 0.015, # 1.5% do CAPEX/ano
            'lcoe_referencia_usd_kwh': 0.043,  # USD/kWh
            'fator_capacidade_medio': 0.18
        },
        'eolico': {
            'capex_usd_kw': 800.0,      # USD/kW (R$ 4.000/kW) - REDUZIDO
            'capex_min_usd_kw': 800.0,   # USD/kW mínimo
            'capex_max_usd_kw': 1200.0,  # USD/kW máximo
            'opex_percent_capex': 0.015, # 1.5% do CAPEX/ano - REDUZIDO
            'lcoe_referencia_usd_kwh': 0.034,  # USD/kWh
            'fator_capacidade_medio': 0.30
        }
    })

    # Perdas do sistema
    perdas_sistema: Dict[str, float] = field(default_factory=lambda: {
        'cabo': 1.5,
        'inversor': 3.0,
        'sombreamento': 2.0,
        'sujeira': 3.0,
        'mismatch': 2.0
    })

### 2. Cache Persistente com SQLite (cache.py)


In [85]:
import os
import json
import time
import pandas as pd

class CachePersistente:

    def __init__(self, base_dir: str = "/content/cache"):
        self.base_dir = base_dir
        os.makedirs(self.base_dir, exist_ok=True)

    def _paths(self, chave: str):
        base = os.path.join(self.base_dir, chave)
        return {
            "parquet": f"{base}.parquet",
            "meta": f"{base}.json"
        }

    def set(self, chave: str, valor: dict, ttl_horas: int):
        paths = self._paths(chave)

        # salva DataFrame em parquet
        valor["dados"].to_parquet(paths["parquet"])

        payload = {
            "meta": valor.get("meta"),
            "fonte": valor.get("fonte"),
            "metadata": valor.get("metadata"),
            "expira_em": time.time() + ttl_horas * 3600
        }

        with open(paths["meta"], "w") as f:
            json.dump(payload, f)

    def get(self, chave: str):
        paths = self._paths(chave)

        if not os.path.exists(paths["meta"]) or not os.path.exists(paths["parquet"]):
            return None

        with open(paths["meta"], "r") as f:
            payload = json.load(f)

        if time.time() > payload["expira_em"]:
            self._delete(paths)
            return None

        df = pd.read_parquet(paths["parquet"])

        return {
            "dados": df,
            "meta": payload.get("meta"),
            "fonte": payload.get("fonte"),
            "metadata": payload.get("metadata")
        }

    def _delete(self, paths: dict):
        for p in paths.values():
            if os.path.exists(p):
                os.remove(p)

### 3. Aquisição de Dados Meteorológicos (dados_meteo.py)


In [86]:
import logging
from typing import Dict, Optional, Tuple
from datetime import datetime, timezone

import pandas as pd
from pvlib.iotools import get_pvgis_hourly, get_nasa_power

logger = logging.getLogger(__name__)

class ColetorDadosMeteorologicos:

    def __init__(self, cache: Optional["CachePersistente"] = None):
        self.cache = cache or CachePersistente()

    def obter_dados_horarios(
        self,
        localizacao: Localizacao,
        periodo: Tuple[str, str],
        usar_cache: bool = True
    ) -> Dict:

        chave_cache = f"meteo_{localizacao.latitude}_{localizacao.longitude}_{periodo[0]}_{periodo[1]}"
        if usar_cache:
            logger.info(f"[CACHE] Checando chave: {chave_cache}")
            cached = self.cache.get(chave_cache)
            if cached:
                logger.info("[CACHE] Dados recuperados do cache")
                return cached

        logger.info(f"[INÍCIO] Coleta de dados meteorológicos para {localizacao.nome} no período {periodo}")

        pvgis_res = self._obter_dados_pvgis(localizacao, periodo)
        nasa_res = self._obter_dados_nasa_power(localizacao, periodo)

        if not pvgis_res and not nasa_res:
            logger.error("[ERRO] Nenhuma fonte retornou dados válidos.")
            raise RuntimeError("Falha total na aquisição de dados meteorológicos")

        # Obter dataframes
        df_pvgis = pvgis_res["dados"] if pvgis_res else pd.DataFrame()
        logger.info(f"[PVGIS] Colunas originais: {list(df_pvgis.columns)}")

        df_nasa = nasa_res["dados"] if nasa_res else pd.DataFrame()
        logger.info(f"[NASA] Colunas originais: {list(df_nasa.columns)}")

        # Consolidar
        logger.info("[CONSOLIDAÇÃO] Combinando PVGIS + NASA POWER")
        df_consolidados = self._consolidar_bases(df_pvgis, df_nasa)

        logger.info(f"[CONSOLIDAÇÃO] Dados consolidados: {df_consolidados.shape}")
        logger.info(f"[CONSOLIDAÇÃO] Colunas: {list(df_consolidados.columns)}")

        resultado = {
            "dados": df_consolidados,
            "metadata": {
                "localizacao": localizacao.nome,
                "periodo": periodo,
                "timestamp_coleta": datetime.now(timezone.utc).isoformat(),
                "fontes": {
                    "pvgis_ok": bool(pvgis_res and not df_pvgis.empty),
                    "nasa_ok": bool(nasa_res and not df_nasa.empty)
                }
            }
        }

        if usar_cache:
            logger.info("[CACHE] Salvando dados no cache")
            self.cache.set(chave_cache, resultado, ttl_horas=24)

        logger.info("[FIM] Dados consolidados prontos")
        return resultado

    def _consolidar_bases(self, pvgis_df: pd.DataFrame, nasa_df: pd.DataFrame) -> pd.DataFrame:
        """
        Agrupa duas bases mantendo todas as colunas:
        - colunas presentes em PVGIS são prioritárias;
        - colunas exclusivas de NASA são preservadas;
        - onde for necessário, a união é por índice horário.
        """

        # Ajustar índices horários
        idx = pvgis_df.index.union(nasa_df.index).sort_values()
        pvgis_df = pvgis_df.reindex(idx)
        nasa_df = nasa_df.reindex(idx)

        # Iniciar df final com todas as colunas de PVGIS
        df_final = pvgis_df.copy()

        # Iterar colunas de NASA POWER
        for col in nasa_df.columns:
            if col in df_final.columns:
                # Campo existe em PVGIS → preenche NaN do PVGIS com dados NASA
                df_final[col] = df_final[col].combine_first(nasa_df[col])
            else:
                # Campo só existe na NASA → anexa
                df_final[col] = nasa_df[col]

        # Opcional: interpolar lacunas
        logger.info("[CONSOLIDAÇÃO] Interpolando dados faltantes, quando aplicável")
        df_final = df_final.sort_index()
        df_final = df_final.interpolate(method="time", limit_direction="both")

        # Algumas validações básicas — apenas se fizer sentido
        # (essa etapa pode ser desativada, depende do seu uso)
        # Exemplo físico para irradiância
        if "ghi" in df_final.columns:
            df_final["ghi"] = df_final["ghi"].clip(lower=0)
        if "dni" in df_final.columns:
            df_final["dni"] = df_final["dni"].clip(lower=0)
        if "dhi" in df_final.columns:
            df_final["dhi"] = df_final["dhi"].clip(lower=0)
        if "wind_speed" in df_final.columns:
            df_final["wind_speed"] = df_final["wind_speed"].clip(lower=0)
        if "PS" in df_final.columns:
            df_final["PS"] = df_final["PS"].clip(lower=0)

        logger.info("[CONSOLIDAÇÃO] Combinação completa de colunas concluída")
        return df_final

    def _obter_dados_pvgis(
        self,
        localizacao: Localizacao,
        periodo: Tuple[str, str]
    ) -> Optional[Dict]:
        logger.info("[PVGIS] Iniciando requisição PVGIS")

        try:
            start_year = int(periodo[0][:4])
            end_year = int(periodo[1][:4])

            data, meta = get_pvgis_hourly(
                latitude=localizacao.latitude,
                longitude=localizacao.longitude,
                start=start_year,
                end=end_year,
                raddatabase="PVGIS-ERA5",
                components=True,
                surface_tilt=0,
                surface_azimuth=180,
                map_variables=True,
                outputformat="json"
            )

            if data.empty:
                logger.warning("[PVGIS] DataFrame vazio")
                return None

            data = data.loc[periodo[0]:periodo[1]]

            logger.info(f"[PVGIS] Linhas obtidas: {len(data)}")
            return {"dados": data, "meta": meta}

        except Exception as e:
            logger.error(f"[PVGIS] Erro ao obter dados: {e}", exc_info=True)
            return None

    def _obter_dados_nasa_power(
        self,
        localizacao: Localizacao,
        periodo: Tuple[str, str]
    ) -> Optional[Dict]:
        logger.info("[NASA] Iniciando requisição NASA POWER")

        try:
            data, meta = get_nasa_power(
                latitude=localizacao.latitude,
                longitude=localizacao.longitude,
                start=periodo[0],
                end=periodo[1],
                parameters=[
                    "ALLSKY_SFC_SW_DWN",
                    "ALLSKY_SFC_SW_DNI",
                    "ALLSKY_SFC_SW_DIFF",
                    "T2M",
                    "WS10M",
                    "PS"
                ],
                community="RE"
            )

            if data.empty:
                logger.warning("[NASA] DataFrame vazio")
                return None

            data = data.loc[periodo[0]:periodo[1]]
            logger.info(f"[NASA] Linhas obtidas: {len(data)}")
            return {"dados": data, "meta": meta}

        except Exception as e:
            logger.error(f"[NASA] Erro ao obter dados: {e}", exc_info=True)
            return None

### 4. Modelagem Solar com pvlib (modelagem_solar.py)


In [87]:
import logging
from typing import Dict, Any

import pandas as pd
import pvlib
from pvlib.modelchain import ModelChain as SolarModelChain
from pvlib.location import Location
from pvlib.pvsystem import PVSystem
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS

logger = logging.getLogger(__name__)

class ModeladorSolar:

    def __init__(self, config: ConfiguracaoSistema):
        self.config = config

    def modelar_sistema(
        self,
        localizacao: Localizacao,
        painel: PainelSolar,
        dados_meteo: Dict[str, Any],
        num_paineis: int = 1,
        tilt: float = 20.0,
        azimuth: float = 180.0
    ) -> Dict[str, Any]:

        logger.info(f"[SOLAR] Modelagem SAPM iniciada para {localizacao.nome}")

        df = dados_meteo.get("dados")
        if df is None or df.empty:
            raise ValueError("Dados meteorológicos vazios")

        df = df.sort_index().ffill().bfill()

        # -----------------------
        # Validações mínimas
        # -----------------------
        for col in ("ghi", "dni", "dhi"):
            if col not in df.columns:
                raise KeyError(f"[SOLAR] Coluna obrigatória ausente: {col}")

        if "temp_air" not in df.columns:
            df["temp_air"] = 25.0
            logger.warning("[SOLAR] temp_air ausente → assumindo 25 °C")

        if "wind_speed" not in df.columns:
            df["wind_speed"] = 1.0
            logger.warning("[SOLAR] wind_speed ausente → assumindo 1 m/s")

        # -----------------------
        # Localização PVlib
        # -----------------------
        location = Location(
            latitude=localizacao.latitude,
            longitude=localizacao.longitude,
            altitude=localizacao.altitude,
            tz=localizacao.timezone
        )

        # -----------------------
        # Sistema FV (SAPM)
        # -----------------------
        system = PVSystem(
            surface_tilt=tilt,
            surface_azimuth=azimuth,
            module_parameters=painel.parametros_pvlib,
            racking_model="open_rack",
            module_type="glass_glass",
            temperature_model_parameters=TEMPERATURE_MODEL_PARAMETERS["sapm"]["open_rack_glass_glass"],
            modules_per_string=num_paineis,
            strings_per_inverter=1,
            inverter_parameters={
                'pdc0': painel.potencia_nominal_w * num_paineis,
                'eta_inv_nom': 0.96
            },
            losses_parameters={'pvwatts_losses': self._calcular_perdas_total()}
        )

        # -----------------------
        # ModelChain CORRETO
        # -----------------------
        mc = SolarModelChain(
            system,
            location,
            dc_model="sapm",
            ac_model="pvwatts",
            temperature_model="sapm",
            losses_model="pvwatts",
            aoi_model="no_loss"

        )

        # -----------------------
        # Entrada climática
        # -----------------------
        logger.info("[SOLAR] Usando GHI/DNI/DHI")

        try:
            weather = pd.DataFrame({
                "ghi": df["ghi"],
                "dni": df["dni"],
                "dhi": df["dhi"],
                "temp_air": df["temp_air"],
                "wind_speed": df["wind_speed"]
            }, index=df.index)

            mc.run_model(weather)

        except Exception as e:
            logger.exception(f"[SOLAR] Falha ao executar ModelChain: {e}")
            raise

        # -----------------------
        # Resultados
        # -----------------------
        if mc.results.ac is None:
            raise RuntimeError("[SOLAR] ModelChain não retornou potência AC")

        # total anual em kWh
        producao_kw = mc.results.ac / 1000.0
        producao_anual_kwh = producao_kw.sum(numeric_only=True)

        # se veio como Series (ex.: MultiArray), extrai escalar
        if isinstance(producao_anual_kwh, pd.Series):
            producao_anual_kwh = producao_anual_kwh.sum()

        capacidade_kw = (painel.potencia_nominal_w * num_paineis) / 1000.0

        fator_capacidade = (
            producao_anual_kwh / (capacidade_kw * len(producao_kw))
            if capacidade_kw > 0 else 0.0
        )

        logger.info(
            "[SOLAR] Produção anual: {:.0f} kWh | FC: {:.3f}".format(
                float(producao_anual_kwh),
                float(fator_capacidade)
            )
        )

        return {
            "producao_horaria_kw": producao_kw,
            "producao_anual_kwh": float(producao_anual_kwh),
            "capacidade_instalada_kw": float(capacidade_kw),
            "fator_capacidade": float(fator_capacidade),
            "metadata": {
                **dados_meteo.get("metadata", {}),
                "num_paineis": num_paineis,
                "modelo": painel.modelo
            }
        }

    def _calcular_perdas_total(self) -> Dict[str, float]:
        """
        Calcula perdas totais do sistema usando o modelo PVWatts.
        Cada componente de perda é fornecido em porcentagem.

        Retorna:
            dict: perdas nos componentes compatíveis com pvlib.pvsystem.pvwatts_losses
        """

        # Exemplo de perdas físicas (em %)
        # ajuste conforme seu projeto e dados típicos de perdas em sistemas fotovoltaicos
        perdas = {
            'soiling': 2.0,                 # sujeira
            'shading': 3.0,                 # sombreamento
            'snow': 0.0,                    # neve (desativado em clima tropical)
            'mismatch': 2.0,                # mismatch elétrico
            'wiring': 1.5,                  # perdas em cabos
            'connections': 0.5,             # ligações
            'lid': 1.5,                     # degradação inicial
            'nameplate_rating': 1.0,        # rating de placa
            'age': 0.0,
            'availability': 3.0,            # disponibilidade / perdas do sistema
        }

        logger.info(f"[SOLAR] Perdas totais configuradas (PVWatts): {perdas}")
        return perdas


### 5. Modelagem Eólica com windpowerlib (modelagem_eolica.py)


In [88]:
import logging
from typing import Dict, Any

import numpy as np
import pandas as pd
from windpowerlib import WindTurbine, ModelChain as WindModelChain

logger = logging.getLogger(__name__)

class ModeladorEolico:

    def __init__(self, config: ConfiguracaoSistema):
        self.config = config

    def modelar_sistema(
        self,
        localizacao: Localizacao,
        turbina: TurbinaEolica,
        dados_meteo: Dict[str, Any],
        num_turbinas: int = 1,
        ref_height: float = 10.0
    ) -> Dict[str, Any]:
        """
        Modelador eólico rigoroso usando windpowerlib com:
        - log-wind profile interno para velocidade
        - densidade via ideal_gas (pressão PS + temp_air)
        - MultiIndex para compatibilidade com ModelChain
        - logs e checagens robustas
        """

        logger.info(f"[EÓLICO] Iniciando modelagem física para {localizacao.nome}")

        df = dados_meteo.get("dados")
        if df is None or df.empty:
            logger.error("[EÓLICO] DataFrame meteorológico vazio ou inválido.")
            raise ValueError("Dados meteorológicos insuficientes.")

        # preencher lacunas fisicamente
        df = df.sort_index().ffill().bfill()

        # checar wind_speed
        if "wind_speed" not in df.columns:
            logger.error("[EÓLICO] 'wind_speed' ausente no DataFrame.")
            raise KeyError("'wind_speed' é requerido.")

        # PRESSÃO (PS: NASA POWER em kPa → Pa)
        if "PS" in df.columns:
            df["pressure_pa"] = df["PS"] * 1000.0
            logger.info("[EÓLICO] Usando PS → pressão atmosférica em Pa.")
        else:
            df["pressure_pa"] = 101325.0
            logger.warning("[EÓLICO] PS ausente → assumindo 101325 Pa (nível do mar).")

        # Temperatura do ar em K (NASA POWER T2M está em °C)
        if "temp_air" in df.columns:
            df["temp_kelvin"] = df["temp_air"] + 273.15
        else:
            df["temp_kelvin"] = 288.15
            logger.warning("[EÓLICO] 'temp_air' ausente → assumindo 15 °C.")

        # densidade do ar via equação dos gases ideais
        # ρ = P/(R*T)
        df["air_density"] = df["pressure_pa"] / (287.0 * df["temp_kelvin"])
        df["air_density"] = df["air_density"].clip(lower=0.8, upper=1.4)

        # construir DataFrame MultiIndex esperado pelo ModelChain
        try:

            weather_df = self._build_weather_multiindex(df, ref_height)

            logger.info("[EÓLICO] Weather_df convertido com sucesso para MultiIndex.")
        except Exception as e:
            logger.error(f"[EÓLICO] Falha ao montar weather_df MultiIndex: {e}")
            raise

        # instanciar turbina
        try:
            turbine = WindTurbine(
                turbine_type=turbina.modelo,
                hub_height=turbina.altura_cubo_m,
                rotor_diameter=turbina.diametro_rotor_m,
                power_curve=turbina.curva_potencia
            )
        except Exception as e:
            logger.exception("[EÓLICO] Erro ao instanciar WindTurbine")
            raise

        # configurar ModelChain com densidade física (ideal_gas)
        try:
            mc = WindModelChain(
                turbine,
                wind_speed_model="logarithmic",
                temperature_model="linear_gradient",
                density_model="ideal_gas",
                power_output_model="power_curve",
                density_correction=True
            )
            mc.run_model(weather_df)  # roda o modelo com os dados
        except Exception as e:
            logger.exception(f"[EÓLICO] Falha ao executar ModelChain: {e}")
            raise

        if mc.power_output is None:
            logger.error("[EÓLICO] ModelChain não retornou potência.")
            raise RuntimeError("Sem produção eólica gerada.")

        # produzir saída em kW
        producao_kw = mc.power_output / 1000.0 * num_turbinas
        producao_anual = producao_kw.sum()
        capacidade_kw = turbina.potencia_nominal_kw * num_turbinas

        horas_totais = len(producao_kw)
        fator_capacidade = (
            producao_anual / (capacidade_kw * horas_totais)
            if capacidade_kw > 0 and horas_totais > 0 else 0.0
        )

        # velocidade no hub (calculada pelo ModelChain)
        try:
            velocidade_media_hub = mc.wind_speed_hub(weather_df).mean()
        except Exception:
            velocidade_media_hub = df["wind_speed"].mean()
            logger.warning("[EÓLICO] Usando média simples de wind_speed para hub.")

        logger.info(
            f"[EÓLICO] Produção anual: {producao_anual:.1f} kWh | "
            f"FC: {fator_capacidade:.3f} | Vento hub médio: {velocidade_media_hub:.2f} m/s"
        )

        return {
            "producao_horaria_kw": producao_kw,
            "producao_anual_kwh": float(producao_anual),
            "capacidade_instalada_kw": float(capacidade_kw),
            "fator_capacidade": float(fator_capacidade),
            "velocidade_media_hub": float(velocidade_media_hub),
            "metadata": {
                **dados_meteo.get("metadata", {}),
                "num_turbinas": num_turbinas,
                "densidade_media": float(df["air_density"].mean())
            }
        }

    # Construir weather_df com MultiIndex (variavel, altura)
    def _build_weather_multiindex(self, df: pd.DataFrame, altura_medicao: float) -> pd.DataFrame:
        """
        Constrói um DataFrame com MultiIndex (variável, altura) para windpowerlib.
        O índice da coluna será:
            ('wind_speed', altura_medicao),
            ('temperature', altura_medicao),
            ('pressure', altura_medicao),
            ('roughness_length', 0)
        """

        # Verificar que df tem as colunas necessárias
        if "wind_speed" not in df.columns:
            raise KeyError("df precisa conter 'wind_speed'")

        # A altura de referência dos dados medidos
        h = float(altura_medicao)

        # Roughness_length é tratada como escalar (0 se não tiver)
        z0 = self.config.perdas_sistema.get("rugosidade", 0.03)

        # Montar o DataFrame MultiIndex
        weather = pd.concat([
            pd.DataFrame({("wind_speed", h): df["wind_speed"]}),
            pd.DataFrame({("temperature", h): df["temp_kelvin"]}),
            pd.DataFrame({("pressure",   h): df["pressure_pa"]}),
            pd.DataFrame({("roughness_length", 0.0): pd.Series(z0, index=df.index)})
        ], axis=1)

        weather.columns = pd.MultiIndex.from_tuples(weather.columns)

        return weather

### 6. Análise Econômica Rigorosa (analise_economica.py)


In [89]:
"""
Análise econômica rigorosa com parâmetros ajustados para 2024.
"""
import logging
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass
import copy

import numpy as np
import pandas as pd
from numpy_financial import irr

logger = logging.getLogger(__name__)

@dataclass
class ResultadosEconomicos:
    """Container para resultados econômicos."""
    lcoe_rpkwh: float  # R$/kWh
    npv_r: float  # R$
    tir_percent: float  # %
    payback_anos: float  # anos
    capex_r: float  # R$
    opex_anual_r: float  # R$/ano
    vida_util: int  # anos
    fluxo_caixa: List[float]  # Fluxo anual

    # Métricas derivadas
    @property
    def roi_percent(self) -> float:
        """Retorno sobre investimento (%)."""
        if self.capex_r > 0:
            return (self.npv_r / self.capex_r) * 100
        return 0.0

    @property
    def custo_energia_evitada(self) -> float:
        """Custo da energia evitada (R$/kWh)."""
        return self.lcoe_rpkwh

class AnalisadorEconomico:
    """
    Analisador econômico com cálculos rigorosos e parâmetros ajustados.
    """

    def __init__(self, config: ConfiguracaoSistema):
        self.config = config
        self.taxa_desconto_real = config.taxa_desconto_real

    def analisar_sistema(
        self,
        tipo: TipoSistema,
        capacidade_kw: float,
        producao_anual_kwh: float,
        vida_util: int,
        tarifa_energia: float = 0.70,  # R$/kWh
        inflacao: float = 0.035,  # 3.5% a.a.
        crescimento_tarifa: float = 0.02  # 2% a.a.
    ) -> ResultadosEconomicos:
        """
        Realiza análise econômica completa com parâmetros realistas.
        """
        logger.info(f"Analisando viabilidade econômica de sistema {tipo.value}")

        try:
            # 1. Calcula custos com valores realistas
            capex_r, opex_anual_r = self._calcular_custos(tipo, capacidade_kw)

            # 2. Calcula fluxo de caixa NOMINAL
            fluxo_caixa_nominal = self._calcular_fluxo_caixa_nominal(
                tipo, capex_r, opex_anual_r, producao_anual_kwh,
                tarifa_energia, vida_util, inflacao, crescimento_tarifa
            )

            # 3. Calcula métricas
            lcoe = self._calcular_lcoe_rigoroso(
                capex_r, opex_anual_r, producao_anual_kwh, vida_util
            )
            npv_val = self._calcular_npv(fluxo_caixa_nominal)
            tir = self._calcular_tir(fluxo_caixa_nominal)
            payback = self._calcular_payback_descontado(fluxo_caixa_nominal)

            resultados = ResultadosEconomicos(
                lcoe_rpkwh=float(lcoe),
                npv_r=float(npv_val),
                tir_percent=float(tir * 100),
                payback_anos=float(payback),
                capex_r=float(capex_r),
                opex_anual_r=float(opex_anual_r),
                vida_util=vida_util,
                fluxo_caixa=fluxo_caixa_nominal
            )

            logger.info(f"LCOE: R$ {lcoe:.3f}/kWh, NPV: R$ {npv_val/1e6:.2f}M, TIR: {tir*100:.1f}%")
            return resultados

        except Exception as e:
            logger.error(f"Erro na análise econômica: {str(e)}", exc_info=True)
            raise

    def _calcular_custos(self, tipo: TipoSistema, capacidade_kw: float) -> Tuple[float, float]:
        """Calcula CAPEX e OPEX com valores realistas para Brasil 2024."""
        params = self.config.parametros_economicos[tipo.value]

        # CAPEX em USD (valores realistas para Brasil)
        capex_usd_kw = params['capex_usd_kw']
        capex_usd = capex_usd_kw * capacidade_kw

        # Converte para BRL
        capex_brl = capex_usd * self.config.taxa_cambio_usd_brl

        # OPEX anual (% do CAPEX) - reduzido para cenário otimista
        opex_percent = params['opex_percent_capex']
        opex_anual_brl = capex_brl * opex_percent

        # Log detalhado
        logger.info(f"Custos {tipo.value}: CAPEX=R${capex_brl/1e6:.2f}M, OPEX=R${opex_anual_brl/1e3:.1f}k/ano")

        return capex_brl, opex_anual_brl

    def _calcular_fluxo_caixa_nominal(
        self,
        tipo: TipoSistema,
        capex_r: float,
        opex_anual_r: float,
        producao_anual_kwh: float,
        tarifa_energia: float,
        vida_util: int,
        inflacao: float,
        crescimento_tarifa: float
    ) -> List[float]:
        """Calcula fluxo de caixa nominal anual com degradação realista."""
        fluxo = [-capex_r]  # Ano 0: investimento

        # Taxa de degradação anual (realista para cada tecnologia)
        degradacao = 0.005 if tipo == TipoSistema.SOLAR else 0.008

        for ano in range(1, vida_util + 1):
            # Produção com degradação
            producao_ano = producao_anual_kwh * (1 - degradacao) ** (ano - 1)

            # Receita com crescimento da tarifa
            tarifa_ano = tarifa_energia * (1 + crescimento_tarifa) ** (ano - 1)
            receita = producao_ano * tarifa_ano

            # OPEX com inflação
            opex_ano = opex_anual_r * (1 + inflacao) ** (ano - 1)

            # Fluxo líquido nominal
            fluxo_ano = receita - opex_ano
            fluxo.append(fluxo_ano)

        return fluxo

    def _calcular_lcoe_rigoroso(
        self,
        capex_r: float,
        opex_anual_r: float,
        producao_anual_kwh: float,
        vida_util: int
    ) -> float:
        """
        Calcula LCOE com fórmula rigorosa incluindo degradação.
        """
        taxa = self.taxa_desconto_real

        # Valor presente dos custos
        vp_custos = capex_r  # CAPEX no ano 0

        # Valor presente do OPEX
        for t in range(1, vida_util + 1):
            opex_t = opex_anual_r * (1 + 0.035) ** (t - 1)  # OPEX com inflação
            vp_custos += opex_t / (1 + taxa) ** t

        # Valor presente da produção (com degradação realista de 0.5% ao ano)
        degradacao = 0.005
        vp_producao = 0
        for t in range(1, vida_util + 1):
            producao_t = producao_anual_kwh * (1 - degradacao) ** (t - 1)
            vp_producao += producao_t / (1 + taxa) ** t

        if vp_producao > 0:
            return vp_custos / vp_producao
        return float('inf')

    def _calcular_npv(self, fluxo_caixa: List[float]) -> float:
        """Calcula NPV com taxa de desconto configurada."""
        npv_val = 0.0
        for t, fluxo in enumerate(fluxo_caixa):
            npv_val += fluxo / (1 + self.taxa_desconto_real) ** t
        return npv_val

    def _calcular_tir(self, fluxo_caixa: List[float]) -> float:
        """Calcula TIR usando numpy_financial.irr."""
        try:
            tir_val = irr(fluxo_caixa)
            if tir_val is None or np.isnan(tir_val):
                return 0.0
            return max(tir_val, 0.0)
        except Exception as e:
            logger.warning(f"Erro no cálculo da TIR: {e}")
            return 0.0

    def _calcular_payback_descontado(self, fluxo_caixa: List[float]) -> float:
        """Calcula payback descontado."""
        investimento = abs(fluxo_caixa[0])
        acumulado = 0.0

        for t, fluxo in enumerate(fluxo_caixa[1:], 1):
            fluxo_descontado = fluxo / (1 + self.taxa_desconto_real) ** t
            acumulado += fluxo_descontado

            if acumulado >= investimento:
                excesso = acumulado - investimento
                fluxo_ano_anterior = fluxo_caixa[t] / (1 + self.taxa_desconto_real) ** t
                fracao = 1 - (excesso / fluxo_ano_anterior) if fluxo_ano_anterior > 0 else 0
                return t - 1 + fracao

        return float(len(fluxo_caixa) - 1)

    def analise_sensibilidade(
        self,
        tipo: TipoSistema,
        capacidade_kw: float,
        producao_anual_kwh: float,
        vida_util: int,
        tarifa_energia: float = 0.70
    ) -> Dict[str, Dict[str, float]]:
        """
        Realiza análise de sensibilidade para cenários otimista/base/pessimista.
        """
        cenarios = {
            'otimista': {'capex_mult': 0.8, 'tarifa': 0.90, 'producao': 1.1},
            'base': {'capex_mult': 1.0, 'tarifa': 0.70, 'producao': 1.0},
            'pessimista': {'capex_mult': 1.2, 'tarifa': 0.50, 'producao': 0.9}
        }

        resultados = {}

        for nome, params in cenarios.items():
            # Configuração temporária
            config_temp = copy.deepcopy(self.config)

            # Ajusta CAPEX
            config_temp.parametros_economicos[tipo.value]['capex_usd_kw'] *= params['capex_mult']

            # Cria analisador temporário
            analisador_temp = AnalisadorEconomico(config_temp)

            # Ajusta produção
            producao_ajustada = producao_anual_kwh * params['producao']

            # Analisa
            resultados_temp = analisador_temp.analisar_sistema(
                tipo, capacidade_kw, producao_ajustada,
                vida_util, params['tarifa']
            )

            resultados[nome] = {
                'lcoe': resultados_temp.lcoe_rpkwh,
                'npv': resultados_temp.npv_r,
                'tir': resultados_temp.tir_percent,
                'payback': resultados_temp.payback_anos,
                'viabilidade': 'alta' if resultados_temp.tir_percent > 12 else
                             'moderada' if resultados_temp.tir_percent > 8 else
                             'baixa'
            }

        return resultados

### 7. Sistema Principal Integrado (sistema_principal.py)


In [90]:
import logging
import json
from typing import Dict, Any, Optional, Tuple
from datetime import datetime

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)


class SistemaAnaliseEnergiaRenovavel:
    """
    Sistema principal com lógica de recomendação aprimorada
    e com ModeladorEolico corrigido integrado.
    """

    def __init__(self, config: Optional[ConfiguracaoSistema] = None):
        self.config = config or ConfiguracaoSistema()

        # Inicializa módulos
        self.coletor = ColetorDadosMeteorologicos()
        self.modelador_solar = ModeladorSolar(self.config)
        self.modelador_eolico = ModeladorEolico(self.config)
        self.analisador_economico = AnalisadorEconomico(self.config)

        logger.info(f"Sistema inicializado: {self.config.nome_projeto}")
        logger.info(f"Taxa desconto: {self.config.taxa_desconto_real}")
        logger.info(f"Taxa câmbio: {self.config.taxa_cambio_usd_brl}")

    def analisar_projeto_completo(
        self,
        localizacao: Localizacao,
        painel: PainelSolar,
        turbina: TurbinaEolica,
        periodo: Tuple[str, str] = ("2023-01-01", "2023-12-31"),
        num_paineis: int = 500,
        num_turbinas: int = 1,
        tarifa_energia: float = 0.70,
        inclinacao_solar: float = 20.0,
        azimuth_solar: float = 180.0,
        gerar_relatorio: bool = True
    ) -> Dict[str, Any]:
        """
        Executa análise completa com lógica de decisão aprimorada.
        """
        logger.info(f"Iniciando análise para {localizacao.nome}")

        try:
            # 1) Dados meteorológicos
            logger.info("Coletando dados meteorológicos...")
            dados_meteo = self.coletor.obter_dados_horarios(localizacao, periodo, usar_cache=False)

            # Extrair média de GHI
            ghi_medio = dados_meteo["dados"].get("ghi", pd.Series(dtype=float)).mean()

            # 2) Modelagem técnica
            logger.info("Modelando sistema solar...")
            resultado_solar = self.modelador_solar.modelar_sistema(
                localizacao,
                painel,
                dados_meteo,
                num_paineis,
                inclinacao_solar,
                azimuth_solar
            )

            logger.info("Modelando sistema eólico...")
            resultado_eolico = self.modelador_eolico.modelar_sistema(
                localizacao,
                turbina,
                dados_meteo,
                num_turbinas
            )

            # 3) Análise econômica
            logger.info("Analisando viabilidade econômica...")
            economico_solar = self.analisador_economico.analisar_sistema(
                TipoSistema.SOLAR,
                resultado_solar["capacidade_instalada_kw"],
                resultado_solar["producao_anual_kwh"],
                painel.vida_util,
                tarifa_energia
            )

            economico_eolico = self.analisador_economico.analisar_sistema(
                TipoSistema.EOLICO,
                resultado_eolico["capacidade_instalada_kw"],
                resultado_eolico["producao_anual_kwh"],
                turbina.vida_util,
                tarifa_energia
            )

            # 4) Análise de sensibilidade
            sensibilidade_solar = self.analisador_economico.analise_sensibilidade(
                TipoSistema.SOLAR,
                resultado_solar["capacidade_instalada_kw"],
                resultado_solar["producao_anual_kwh"],
                painel.vida_util,
                tarifa_energia
            )

            sensibilidade_eolica = self.analisador_economico.analise_sensibilidade(
                TipoSistema.EOLICO,
                resultado_eolico["capacidade_instalada_kw"],
                resultado_eolico["producao_anual_kwh"],
                turbina.vida_util,
                tarifa_energia
            )

            # 5) Comparação técnica & econômica
            comparacao = self._comparar_sistemas(
                resultado_solar,
                resultado_eolico,
                economico_solar,
                economico_eolico,
                ghi_medio,
                resultado_eolico.get("velocidade_media_hub", 0.0)
            )

            # 6) Recomendação
            recomendacao = self._gerar_recomendacao(
                comparacao,
                ghi_medio,
                resultado_eolico.get("velocidade_media_hub", 0.0)
            )

            # 7) Compilar resultados
            resultados = {
                "metadados": {
                    "data_analise": datetime.now().isoformat(),
                    "versao_sistema": "5.0",
                    "localizacao": localizacao.__dict__,
                    "periodo_analise": periodo,
                    "ghi_medio_wm2": ghi_medio
                },
                "solar": {
                    "tecnicos": resultado_solar,
                    "economicos": economico_solar,
                    "sensibilidade": sensibilidade_solar
                },
                "eolica": {
                    "tecnicos": resultado_eolico,
                    "economicos": economico_eolico,
                    "sensibilidade": sensibilidade_eolica
                },
                "comparacao": comparacao,
                "recomendacao": recomendacao,
                "resumo_executivo": self._gerar_resumo_executivo(
                    resultado_solar,
                    resultado_eolico,
                    economico_solar,
                    economico_eolico,
                    recomendacao,
                    ghi_medio,
                    resultado_eolico.get("velocidade_media_hub", 0.0)
                )
            }

            if gerar_relatorio:
                self._gerar_relatorio(resultados, localizacao.nome)

            logger.info("Análise concluída com sucesso")
            return resultados

        except Exception as e:
            logger.error(f"Erro na análise completa: {str(e)}", exc_info=True)
            raise

    # -----------------------------------------------------
    # Comparação técnica & econômica (pontuação robusta)
    # -----------------------------------------------------

    def _comparar_sistemas(
        self,
        solar_tec: Dict[str, Any],
        eolica_tec: Dict[str, Any],
        solar_econ: ResultadosEconomicos,
        eolica_econ: ResultadosEconomicos,
        ghi_medio: float,
        velocidade_media_hub: float
    ) -> Dict[str, Any]:

        # MÉTRICAS TÉCNICAS
        fc_solar = solar_tec.get("fator_capacidade", 0.0)
        fc_eolico = eolica_tec.get("fator_capacidade", 0.0)

        prod_solar = solar_tec.get("producao_anual_kwh", 0.0)
        prod_eolico = eolica_tec.get("producao_anual_kwh", 0.0)

        # MÉTRICAS ECONÔMICAS
        lcoe_solar = solar_econ.lcoe_rpkwh
        lcoe_eolico = eolica_econ.lcoe_rpkwh

        npv_solar = solar_econ.npv_r
        npv_eolico = eolica_econ.npv_r

        # Normalizações
        max_fc = max(fc_solar, fc_eolico, 1e-6)
        norm_fc_solar = fc_solar / max_fc
        norm_fc_eolico = fc_eolico / max_fc

        max_prod = max(prod_solar, prod_eolico, 1e-6)
        norm_prod_solar = prod_solar / max_prod
        norm_prod_eolico = prod_eolico / max_prod

        max_lcoe = max(lcoe_solar, lcoe_eolico, 1e-6)
        score_lcoe_solar = (max_lcoe - lcoe_solar) / max_lcoe
        score_lcoe_eolico = (max_lcoe - lcoe_eolico) / max_lcoe

        max_npv = max(abs(npv_solar), abs(npv_eolico), 1e-6)
        score_npv_solar = npv_solar / max_npv
        score_npv_eolico = npv_eolico / max_npv

        return {
            "tecnico": {
                "norm_fc": {"solar": norm_fc_solar, "eolica": norm_fc_eolico},
                "norm_prod": {"solar": norm_prod_solar, "eolica": norm_prod_eolico},
                "recurso": {"ghi": ghi_medio, "vento_hub": velocidade_media_hub},
            },
            "economico": {
                "score_lcoe": {"solar": score_lcoe_solar, "eolica": score_lcoe_eolico},
                "score_npv": {"solar": score_npv_solar, "eolica": score_npv_eolico},
            },
        }

    # -----------------------------------------------------
    # Recomendação com multicritério balanceado
    # -----------------------------------------------------

    def _gerar_recomendacao(
        self,
        comparacao: Dict[str, Any],
        ghi_medio: float,
        velocidade_media_hub: float
    ) -> Dict[str, Any]:

        tec = comparacao["tecnico"]
        econ = comparacao["economico"]

        # Regra híbrida prioritária
        if ghi_medio > 200 and velocidade_media_hub > 4.5:
            sistema = TipoSistema.HIBRIDO
            regra = "H1"
            justificativa = "Ambos os recursos são fortes — híbrido recomendado."

            score_solar = tec["norm_fc"]["solar"] + tec["norm_prod"]["solar"] + econ["score_lcoe"]["solar"] + econ["score_npv"]["solar"]
            score_eolico = tec["norm_fc"]["eolica"] + tec["norm_prod"]["eolica"] + econ["score_lcoe"]["eolica"] + econ["score_npv"]["eolica"]

        else:
            # Score total
            score_solar = (
                tec["norm_fc"]["solar"] + tec["norm_prod"]["solar"]
                + econ["score_lcoe"]["solar"] + econ["score_npv"]["solar"]
            )
            score_eolico = (
                tec["norm_fc"]["eolica"] + tec["norm_prod"]["eolica"]
                + econ["score_lcoe"]["eolica"] + econ["score_npv"]["eolica"]
            )

            score_solar += ghi_medio / 300
            score_eolico += velocidade_media_hub / 10

            if score_solar > score_eolico:
                sistema = TipoSistema.SOLAR
                regra = "D1"
                justificativa = "Score total favorece solar."
            else:
                sistema = TipoSistema.EOLICO
                regra = "D2"
                justificativa = "Score total favorece eólico."

        return {
            "sistema_recomendado": sistema.value,
            "justificativa": justificativa,
            "regra_aplicada": regra,
            "pontuacao_solar": score_solar,
            "pontuacao_eolica": score_eolico
        }


    # -----------------------------------------------------
    # Resumo executivo
    # -----------------------------------------------------

    def _gerar_resumo_executivo(
        self,
        solar_tec: Dict[str, Any],
        eolica_tec: Dict[str, Any],
        solar_econ: ResultadosEconomicos,
        eolica_econ: ResultadosEconomicos,
        recomendacao: Dict[str, Any],
        ghi_medio: float,
        velocidade_media_hub: float
    ) -> Dict[str, Any]:

        sistema = recomendacao["sistema_recomendado"]

        if sistema == TipoSistema.SOLAR.value:
            escolhido_tec = solar_tec
            escolhido_econ = solar_econ

        elif sistema == TipoSistema.EOLICO.value:
            escolhido_tec = eolica_tec
            escolhido_econ = eolica_econ

        else:
            # cenário híbrido: criar objeto econômico com médias
            class DadosHibridos:
                def __init__(self):
                    self.lcoe_rpkwh = 0.0
                    self.npv_r = 0.0
                    self.tir_percent = 0.0
                    self.payback_anos = 0.0
                    self.capex_r = 0.0
                    self.opex_anual_r = 0.0

            escolhido_tec = {
                'producao_anual_kwh': (solar_tec['producao_anual_kwh'] + eolica_tec['producao_anual_kwh']) / 2,
                'capacidade_instalada_kw': (solar_tec['capacidade_instalada_kw'] + eolica_tec['capacidade_instalada_kw']) / 2,
                'fator_capacidade': (solar_tec['fator_capacidade'] + eolica_tec['fator_capacidade']) / 2,
            }

            escolhido_econ = DadosHibridos()
            escolhido_econ.lcoe_rpkwh = (solar_econ.lcoe_rpkwh + eolica_econ.lcoe_rpkwh) / 2
            escolhido_econ.npv_r = (solar_econ.npv_r + eolica_econ.npv_r) / 2
            escolhido_econ.tir_percent = (solar_econ.tir_percent + eolica_econ.tir_percent) / 2
            escolhido_econ.payback_anos = (solar_econ.payback_anos + eolica_econ.payback_anos) / 2
            escolhido_econ.capex_r = (solar_econ.capex_r + eolica_econ.capex_r) / 2
            escolhido_econ.opex_anual_r = (solar_econ.opex_anual_r + eolica_econ.opex_anual_r) / 2


        fator_capacidade = escolhido_tec["producao_anual_kwh"] / (escolhido_tec["capacidade_instalada_kw"] * 8760 + 1e-6)

        return {
            "sistema_recomendado": sistema,
            "justificativa": recomendacao["justificativa"],
            "principais_metricas": {
                "lcoe_rpkwh": escolhido_econ.lcoe_rpkwh,
                "npv_r": escolhido_econ.npv_r,
                "npv_milhoes": escolhido_econ.npv_r / 1e6,
                "payback_anos": getattr(escolhido_econ, "payback_anos", None),
                "investimento_milhoes": escolhido_econ.capex_r / 1e6,
                "producao_anual_kwh": escolhido_tec["producao_anual_kwh"],
                "producao_gwh": escolhido_tec["producao_anual_kwh"] / 1e6,
                "capacidade_kw": escolhido_tec["capacidade_instalada_kw"],
                "capacidade_mw": escolhido_tec["capacidade_instalada_kw"] / 1000,
                "fator_capacidade": fator_capacidade,
                "tir_percent": getattr(escolhido_econ, "tir", None) * 100 if hasattr(escolhido_econ, "tir") else None,
                "ghi_medio_wm2": ghi_medio,
                "vento_media_hub_ms": velocidade_media_hub
            }

        }

    def _preparar_para_json(self, obj: Any) -> Any:
        if isinstance(obj, dict):
            return {k: self._preparar_para_json(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [self._preparar_para_json(item) for item in obj]
        elif isinstance(obj, (pd.DataFrame, pd.Series)):
            return obj.to_dict()
        elif isinstance(obj, (datetime, pd.Timestamp)):
            return obj.isoformat()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (str, int, float, bool, type(None))):
            return obj
        elif hasattr(obj, "__dict__"):
            return {k: self._preparar_para_json(v) for k, v in obj.__dict__.items() if not k.startswith("_")}
        else:
            return str(obj)

    def _gerar_relatorio(self, resultados: Dict[str, Any], local_nome: str):
        filename = f"relatorio_{local_nome.replace(' ','_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(self._preparar_para_json(resultados), f, indent=2, ensure_ascii=False)
        logger.info(f"Relatório salvo: {filename}")

### 9. Exemplo de Uso (main.py)


In [91]:
"""
Análise Comparativa de 15 Locais Brasileiros - Versão 4.1
"""
import logging
from datetime import datetime
import pandas as pd
import os
import sys

def configurar_logging(nome_analise: str):
    """Configura logging para a análise."""
    log_dir = "/tmp/logs"
    os.makedirs(log_dir, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"{nome_analise}_{timestamp}.log")

    # Configurar logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file, encoding='utf-8'),
            logging.StreamHandler()
        ],
        force=True  # força reconfiguração, necessário no Google Colab
    )


    return logging.getLogger(nome_analise), log_file

def executar_analise_comparativa():
    """Executa análise comparativa para os 15 locais."""

    logger, log_file = configurar_logging("analise_comparativa_15_locais")

    print("\n" + "="*100)
    print("ANÁLISE COMPARATIVA DE 15 LOCAIS BRASILEIROS - VERSÃO 4.1")
    print("="*100)

    logger.info("="*80)
    logger.info("INICIANDO ANÁLISE COMPARATIVA DE 15 LOCAIS")
    logger.info("="*80)

    resultados_comparativos = []

    try:
        # 1. Configuração do sistema
        logger.info("1. Configurando sistema...")
        print("\n1. Configurando sistema...")

        config = ConfiguracaoSistema()
        sistema = SistemaAnaliseEnergiaRenovavel(config)

        # 2. Definição dos locais
        logger.info("2. Definindo os 15 locais para análise...")
        print("2. Definindo os 15 locais para análise...")

        locais = [
            # ☀️ Locais com Alto Potencial Solar
            {"nome": "Barreiras, BA", "tipo": "solar", "lat": -12.152, "lon": -44.990},
            {"nome": "Jaíba, MG", "tipo": "solar", "lat": -15.343, "lon": -43.672},
            {"nome": "São Desidério, BA", "tipo": "solar", "lat": -12.357, "lon": -44.976},
            {"nome": "Caldas Novas, GO", "tipo": "solar", "lat": -17.744, "lon": -48.625},
            {"nome": "Teresina, PI", "tipo": "solar", "lat": -5.089, "lon": -42.802},

            # 💨 Locais com Alto Potencial Eólico
            {"nome": "Osório, RS", "tipo": "eolico", "lat": -29.886, "lon": -50.269},
            {"nome": "Acaraú, CE", "tipo": "eolico", "lat": -2.887, "lon": -40.118},
            {"nome": "Chapada do Araripe, PE", "tipo": "eolico", "lat": -7.576, "lon": -40.498},
            {"nome": "Mucuripe, CE", "tipo": "eolico", "lat": -3.718, "lon": -38.543},
            {"nome": "São João do Cariri, PB", "tipo": "eolico", "lat": -7.387, "lon": -36.534},

            # 🔄 Locais com Potencial Híbrido
            {"nome": "Morro do Chapéu, BA", "tipo": "hibrido", "lat": -11.548, "lon": -41.158},
            {"nome": "Natal, RN", "tipo": "hibrido", "lat": -5.779, "lon": -35.200},
            {"nome": "Xique-Xique, BA", "tipo": "hibrido", "lat": -10.823, "lon": -42.730},
            {"nome": "João Câmara, RN", "tipo": "hibrido", "lat": -5.540, "lon": -35.819},
            {"nome": "Porto Alegre, RS", "tipo": "hibrido", "lat": -30.034, "lon": -51.217}
        ]

        # 3. Equipamentos padrão
        logger.info("3. Configurando equipamentos padrão...")
        print("3. Configurando equipamentos padrão...")

        painel = PainelSolar()
        turbina = TurbinaEolica()

        # 4. Executar análises
        logger.info("4. Executando análises individuais...")
        print("4. Executando análises individuais...")
        print("-" * 80)

        for i, local in enumerate(locais, 1):
            logger.info(f"[{i}/15] Iniciando análise para: {local['nome']} ({local['tipo'].upper()})")
            print(f"\n[{i}/15] Analisando: {local['nome']} ({local['tipo'].upper()})")

            try:
                localizacao = Localizacao(
                    latitude=local['lat'],
                    longitude=local['lon'],
                    nome=local['nome']
                )

                resultados = sistema.analisar_projeto_completo(
                    localizacao=localizacao,
                    painel=painel,
                    turbina=turbina,
                    periodo=("2023-01-01", "2023-12-31"),
                    num_paineis=500,
                    num_turbinas=1,
                    tarifa_energia=0.75,
                    gerar_relatorio=False
                )

                resumo = resultados['resumo_executivo']
                recomendacao = resultados['recomendacao']

               # VIABILIDADE do cenário base
                if recomendacao['sistema_recomendado'] == 'solar':
                    viab_base = resultados['solar']['sensibilidade']['base']['viabilidade']
                    lcoe_base = resultados['solar']['sensibilidade']['base']['lcoe']
                    tir_base = resultados['solar']['sensibilidade']['base']['tir']
                    payback_base = resultados['solar']['sensibilidade']['base']['payback']
                elif recomendacao['sistema_recomendado'] == 'eolico':
                    viab_base = resultados['eolica']['sensibilidade']['base']['viabilidade']
                    lcoe_base = resultados['eolica']['sensibilidade']['base']['lcoe']
                    tir_base = resultados['eolica']['sensibilidade']['base']['tir']
                    payback_base = resultados['eolica']['sensibilidade']['base']['payback']
                else:
                    # híbrido — média (ou pode ser outra regra que desejar)
                    viab_base = (
                        resultados['solar']['sensibilidade']['base']['viabilidade'],
                        resultados['eolica']['sensibilidade']['base']['viabilidade']
                    )
                    lcoe_base = (
                        resultados['solar']['sensibilidade']['base']['lcoe'],
                        resultados['eolica']['sensibilidade']['base']['lcoe']
                    )
                    tir_base = (
                        resultados['solar']['sensibilidade']['base']['tir'],
                        resultados['eolica']['sensibilidade']['base']['tir']
                    )
                    payback_base = (
                        resultados['solar']['sensibilidade']['base']['payback'],
                        resultados['eolica']['sensibilidade']['base']['payback']
                    )

                resultado_local = {
                    'local': local['nome'],
                    'tipo_esperado': local['tipo'],
                    'recomendacao_algoritmo': recomendacao['sistema_recomendado'],
                    'concordancia_tipo': local['tipo'] == recomendacao['sistema_recomendado'],
                    'regra_aplicada': recomendacao['regra_aplicada'],

                    # Econômicos (base)
                    'lcoe_rpkwh': resumo['principais_metricas']['lcoe_rpkwh'],
                    'npv_milhoes': resumo['principais_metricas']['npv_r'] / 1e6,
                    'payback_anos': resumo['principais_metricas']['payback_anos'],
                    'tir_percent': resumo['principais_metricas']['tir_percent'],
                    'investimento_milhoes': (
                        resumo['principais_metricas']['capacidade_kw'] *
                        (resultados['solar']['economicos'].capex_r
                        if recomendacao['sistema_recomendado'] == 'solar'
                        else resultados['eolica']['economicos'].capex_r)
                    ) / 1e6,

                    # Técnicos
                    'producao_gwh': resumo['principais_metricas']['producao_anual_kwh'] / 1e6,
                    'capacidade_mw': resumo['principais_metricas']['capacidade_kw'] / 1000,
                    'fator_capacidade': resumo['principais_metricas']['fator_capacidade'],

                    # Recursos
                    'ghi_medio': resumo['principais_metricas']['ghi_medio_wm2'],
                    'vento_medio': resumo['principais_metricas']['vento_media_hub_ms'],

                    # VIABILIDADE: cenário base a partir da função de sensibilidade
                    'viabilidade': viab_base,
                    'lcoe_base_sens': lcoe_base,
                    'tir_base_sens': tir_base,
                    'payback_base_sens': payback_base,

                    # Sensibilidade completa
                    'sens_otimista_lcoe': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['otimista']['lcoe'],
                    'sens_base_lcoe': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['base']['lcoe'],
                    'sens_pessimista_lcoe': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['pessimista']['lcoe'],

                    'sens_otimista_npv': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['otimista']['npv'],
                    'sens_base_npv': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['base']['npv'],
                    'sens_pessimista_npv': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['pessimista']['npv'],

                    'sens_otimista_tir': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['otimista']['tir'],
                    'sens_base_tir': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['base']['tir'],
                    'sens_pessimista_tir': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['pessimista']['tir'],

                    'sens_otimista_payback': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['otimista']['payback'],
                    'sens_base_payback': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['base']['payback'],
                    'sens_pessimista_payback': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['pessimista']['payback'],

                    'sens_otimista_viabilidade': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['otimista']['viabilidade'],
                    'sens_base_viabilidade': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['base']['viabilidade'],
                    'sens_pessimista_viabilidade': resultados['solar' if recomendacao['sistema_recomendado'] == 'solar' else 'eolica']['sensibilidade']['pessimista']['viabilidade']
                }

                resultados_comparativos.append(resultado_local)

                # Feedback
                concord_icon = "✅" if resultado_local['concordancia_tipo'] else "❌"
                viab_icon = "✅" if resultado_local['viabilidade'] == 'alta' else "⚠️"

                mensagem = f"{viab_icon} {recomendacao['sistema_recomendado'].upper()} | LCOE: R${resultado_local['lcoe_rpkwh']:.3f}/kWh"
                logger.info(mensagem)
                print(f"   {mensagem}")

                print(f"   Concordância: {concord_icon} (Esperado: {local['tipo']}, Algoritmo: {recomendacao['sistema_recomendado']})")
                print(f"   Regra aplicada: {recomendacao['regra_aplicada']}")

            except Exception as e:
                error_msg = f"Erro em {local['nome']}: {str(e)}"
                logger.error(error_msg, exc_info=True)
                print(f"   ❌ Erro: {str(e)[:50]}...")

                resultados_comparativos.append({
                    'local': local['nome'],
                    'tipo_esperado': local['tipo'],
                    'erro': str(e)
                })

        # 5. Gerar relatório final
        logger.info("5. Gerando relatório comparativo...")
        print("\n" + "="*100)
        print("5. Gerando relatório comparativo...")

        if resultados_comparativos:
            df_resultados = pd.DataFrame([r for r in resultados_comparativos if 'erro' not in r])

            if not df_resultados.empty:
                # Estatísticas
                concordancia = df_resultados['concordancia_tipo'].mean() * 100
                lcoe_medio = df_resultados['lcoe_rpkwh'].mean()
                viabilidade_alta = (df_resultados['viabilidade'] == 'alta_viabilidade').mean() * 100

                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"resultados_comparativos_15_locais_{timestamp}.csv"
                df_resultados.to_csv(filename, index=False, encoding='utf-8-sig')

                # Relatório
                print("\n" + "="*100)
                print("RESUMO EXECUTIVO - ANÁLISE COMPARATIVA")
                print("="*100)
                print(f"📊 Taxa de concordância: {concordancia:.1f}%")
                print(f"💰 LCOE médio: R$ {lcoe_medio:.3f}/kWh")
                print(f"✅ Viabilidade alta: {viabilidade_alta:.1f}%")
                print(f"📁 Resultados salvos em: {filename}")
                print(f"📋 Logs salvos em: {log_file}")

                # Detalhar discordâncias
                discordantes = df_resultados[~df_resultados['concordancia_tipo']]
                if not discordantes.empty:
                    print("\n📍 LOCAIS COM DISCORDÂNCIA:")
                    for _, row in discordantes.iterrows():
                        print(f"   • {row['local']}: Esperado {row['tipo_esperado']}, "
                              f"Recomendado {row['recomendacao_algoritmo']} (Regra {row['regra_aplicada']})")

        logger.info("Análise comparativa concluída!")
        print("\n" + "="*100)
        print("✅ ANÁLISE COMPARATIVA CONCLUÍDA!")
        print("="*100)

        return resultados_comparativos

    except Exception as e:
        logger.critical(f"ERRO CRÍTICO: {str(e)}", exc_info=True)
        print(f"\n❌ ERRO CRÍTICO: {str(e)}")
        return None

if __name__ == "__main__":
    executar_analise_comparativa()

2026-01-13 15:01:35,377 - analise_comparativa_15_locais - INFO - ================================================================================
2026-01-13 15:01:35,383 - analise_comparativa_15_locais - INFO - INICIANDO ANÁLISE COMPARATIVA DE 15 LOCAIS
2026-01-13 15:01:35,385 - analise_comparativa_15_locais - INFO - ================================================================================
2026-01-13 15:01:35,387 - analise_comparativa_15_locais - INFO - 1. Configurando sistema...
2026-01-13 15:01:35,392 - __main__ - INFO - Sistema inicializado: Análise Comparativa 15 Locais
2026-01-13 15:01:35,393 - __main__ - INFO - Taxa desconto: 0.06
2026-01-13 15:01:35,397 - __main__ - INFO - Taxa câmbio: 5.0
2026-01-13 15:01:35,398 - analise_comparativa_15_locais - INFO - 2. Definindo os 15 locais para análise...
2026-01-13 15:01:35,399 - analise_comparativa_15_locais - INFO - 3. Configurando equipamentos padrão...
2026-01-13 15:01:35,407 - analise_comparativa_15_locais - INFO - 4. Executan


ANÁLISE COMPARATIVA DE 15 LOCAIS BRASILEIROS - VERSÃO 4.1

1. Configurando sistema...
2. Definindo os 15 locais para análise...
3. Configurando equipamentos padrão...
4. Executando análises individuais...
--------------------------------------------------------------------------------

[1/15] Analisando: Barreiras, BA (SOLAR)


2026-01-13 15:01:38,168 - __main__ - INFO - [PVGIS] Linhas obtidas: 8760
2026-01-13 15:01:38,174 - __main__ - INFO - [NASA] Iniciando requisição NASA POWER
2026-01-13 15:01:44,147 - __main__ - INFO - [NASA] Linhas obtidas: 8760
2026-01-13 15:01:44,153 - __main__ - INFO - [PVGIS] Colunas originais: ['poa_direct', 'poa_sky_diffuse', 'poa_ground_diffuse', 'solar_elevation', 'temp_air', 'wind_speed', 'Int']
2026-01-13 15:01:44,155 - __main__ - INFO - [NASA] Colunas originais: ['ghi', 'dni', 'dhi', 'temp_air', 'wind_speed', 'PS']
2026-01-13 15:01:44,163 - __main__ - INFO - [CONSOLIDAÇÃO] Combinando PVGIS + NASA POWER
2026-01-13 15:01:44,219 - __main__ - INFO - [CONSOLIDAÇÃO] Interpolando dados faltantes, quando aplicável
2026-01-13 15:01:44,359 - __main__ - INFO - [CONSOLIDAÇÃO] Combinação completa de colunas concluída
2026-01-13 15:01:44,364 - __main__ - INFO - [CONSOLIDAÇÃO] Dados consolidados: (17520, 11)
2026-01-13 15:01:44,369 - __main__ - INFO - [CONSOLIDAÇÃO] Colunas: ['poa_direct', 

   ✅ SOLAR | LCOE: R$0.286/kWh
   Concordância: ✅ (Esperado: solar, Algoritmo: solar)
   Regra aplicada: D1

[2/15] Analisando: Jaíba, MG (SOLAR)


2026-01-13 15:01:47,714 - __main__ - INFO - [PVGIS] Linhas obtidas: 8760
2026-01-13 15:01:47,720 - __main__ - INFO - [NASA] Iniciando requisição NASA POWER
2026-01-13 15:01:53,057 - __main__ - INFO - [NASA] Linhas obtidas: 8760
2026-01-13 15:01:53,058 - __main__ - INFO - [PVGIS] Colunas originais: ['poa_direct', 'poa_sky_diffuse', 'poa_ground_diffuse', 'solar_elevation', 'temp_air', 'wind_speed', 'Int']
2026-01-13 15:01:53,060 - __main__ - INFO - [NASA] Colunas originais: ['ghi', 'dni', 'dhi', 'temp_air', 'wind_speed', 'PS']
2026-01-13 15:01:53,061 - __main__ - INFO - [CONSOLIDAÇÃO] Combinando PVGIS + NASA POWER
2026-01-13 15:01:53,073 - __main__ - INFO - [CONSOLIDAÇÃO] Interpolando dados faltantes, quando aplicável
2026-01-13 15:01:53,101 - __main__ - INFO - [CONSOLIDAÇÃO] Combinação completa de colunas concluída
2026-01-13 15:01:53,102 - __main__ - INFO - [CONSOLIDAÇÃO] Dados consolidados: (17520, 11)
2026-01-13 15:01:53,103 - __main__ - INFO - [CONSOLIDAÇÃO] Colunas: ['poa_direct', 

...

   ⚠️ HIBRIDO | LCOE: R$0.260/kWh
   Concordância: ✅ (Esperado: hibrido, Algoritmo: hibrido)
   Regra aplicada: H1

[14/15] Analisando: João Câmara, RN (HIBRIDO)


2026-01-13 15:03:23,271 - __main__ - INFO - [PVGIS] Linhas obtidas: 8760
2026-01-13 15:03:23,272 - __main__ - INFO - [NASA] Iniciando requisição NASA POWER
2026-01-13 15:03:29,073 - __main__ - INFO - [NASA] Linhas obtidas: 8760
2026-01-13 15:03:29,074 - __main__ - INFO - [PVGIS] Colunas originais: ['poa_direct', 'poa_sky_diffuse', 'poa_ground_diffuse', 'solar_elevation', 'temp_air', 'wind_speed', 'Int']
2026-01-13 15:03:29,075 - __main__ - INFO - [NASA] Colunas originais: ['ghi', 'dni', 'dhi', 'temp_air', 'wind_speed', 'PS']
2026-01-13 15:03:29,078 - __main__ - INFO - [CONSOLIDAÇÃO] Combinando PVGIS + NASA POWER
2026-01-13 15:03:29,088 - __main__ - INFO - [CONSOLIDAÇÃO] Interpolando dados faltantes, quando aplicável
2026-01-13 15:03:29,115 - __main__ - INFO - [CONSOLIDAÇÃO] Combinação completa de colunas concluída
2026-01-13 15:03:29,116 - __main__ - INFO - [CONSOLIDAÇÃO] Dados consolidados: (17520, 11)
2026-01-13 15:03:29,119 - __main__ - INFO - [CONSOLIDAÇÃO] Colunas: ['poa_direct', 

   ⚠️ HIBRIDO | LCOE: R$0.207/kWh
   Concordância: ✅ (Esperado: hibrido, Algoritmo: hibrido)
   Regra aplicada: H1

[15/15] Analisando: Porto Alegre, RS (HIBRIDO)


2026-01-13 15:03:31,545 - __main__ - INFO - [PVGIS] Linhas obtidas: 8760
2026-01-13 15:03:31,546 - __main__ - INFO - [NASA] Iniciando requisição NASA POWER
2026-01-13 15:03:36,983 - __main__ - INFO - [NASA] Linhas obtidas: 8760
2026-01-13 15:03:36,984 - __main__ - INFO - [PVGIS] Colunas originais: ['poa_direct', 'poa_sky_diffuse', 'poa_ground_diffuse', 'solar_elevation', 'temp_air', 'wind_speed', 'Int']
2026-01-13 15:03:36,985 - __main__ - INFO - [NASA] Colunas originais: ['ghi', 'dni', 'dhi', 'temp_air', 'wind_speed', 'PS']
2026-01-13 15:03:36,986 - __main__ - INFO - [CONSOLIDAÇÃO] Combinando PVGIS + NASA POWER
2026-01-13 15:03:37,003 - __main__ - INFO - [CONSOLIDAÇÃO] Interpolando dados faltantes, quando aplicável
2026-01-13 15:03:37,049 - __main__ - INFO - [CONSOLIDAÇÃO] Combinação completa de colunas concluída
2026-01-13 15:03:37,051 - __main__ - INFO - [CONSOLIDAÇÃO] Dados consolidados: (17520, 11)
2026-01-13 15:03:37,052 - __main__ - INFO - [CONSOLIDAÇÃO] Colunas: ['poa_direct', 

   ✅ EOLICO | LCOE: R$0.348/kWh
   Concordância: ❌ (Esperado: hibrido, Algoritmo: eolico)
   Regra aplicada: D2

5. Gerando relatório comparativo...

RESUMO EXECUTIVO - ANÁLISE COMPARATIVA
📊 Taxa de concordância: 60.0%
💰 LCOE médio: R$ 0.251/kWh
✅ Viabilidade alta: 0.0%
📁 Resultados salvos em: resultados_comparativos_15_locais_20260113_150337.csv
📋 Logs salvos em: /tmp/logs/analise_comparativa_15_locais_20260113_150135.log

📍 LOCAIS COM DISCORDÂNCIA:
   • Acaraú, CE: Esperado eolico, Recomendado hibrido (Regra H1)
   • Chapada do Araripe, PE: Esperado eolico, Recomendado hibrido (Regra H1)
   • Mucuripe, CE: Esperado eolico, Recomendado hibrido (Regra H1)
   • São João do Cariri, PB: Esperado eolico, Recomendado hibrido (Regra H1)
   • Morro do Chapéu, BA: Esperado hibrido, Recomendado eolico (Regra D2)
   • Porto Alegre, RS: Esperado hibrido, Recomendado eolico (Regra D2)

✅ ANÁLISE COMPARATIVA CONCLUÍDA!
